## 📐 **Project Summary:** Misconception Classification in Math Explanations

---

### **Goal and Approach**

The core objective of this project was to build a classification system capable of analyzing student-generated math explanations to predict two key outcomes:

1.  **Correctness:** Whether the explanation leads to a correct or incorrect final answer.
2.  **Misconception Type:** The specific type of mathematical or conceptual error present in the explanation (e.g., 'Subtraction', 'Inversion').

The project utilized a foundation dataset from a Kaggle competition focused on student math misunderstandings and employed **two separate specialized models** rather than a single multi-task model.

---

### **Data Challenges & Augmentation**

The original data suffered from **limited sample variety**, featuring a restricted set of question templates and fixed numerical values. This limitation meant any trained model would struggle to generalize to questions with different numerical combinations.

To overcome this, a comprehensive **data augmentation strategy** was implemented:
* **Template Generation:** A variety of templates were built to preserve the fixed linguistic structure and semantics of the original questions while **varying the numerical values** (e.g., replacing 1/3 × 2 with 1/4 × 3).
* **Misconception Coverage:** Templates were designed to cover all existing misconception classes and generate numerous correct samples.
* **Multi-Label Samples:** Templates were created to synthesize samples exhibiting **multiple co-occurring misconceptions** (e.g., both 'Subtraction' and 'Inversion' being marked).

This augmented dataset, combined with the original Kaggle data, was used for both model training and validation.

---

### **Feature Engineering: Mathematical Context** 🔑

A critical step in enabling the model to accurately classify misconceptions was the creation of a **comprehensive set of mathematical context features**. These features allow the language model to interpret the numbers and operations mentioned in the student's *explanation* by providing the relevant numerical relationships present in the *question*.

This feature engineering process involved extracting every number and fraction from the question and generating all possible associated mathematical contexts, including:

* **Pre-computed Misconception Answers:** Calculating values that would result from common errors (e.g., the result of dividing a number by a fraction *in the reverse order* to catch **Swap Dividend** errors).
* **Fraction & Decimal Analysis:** Computing the **Least Common Multiple (LCM)** and **Greatest Common Divisor (GCD)** of denominators, identifying complementary fractions, and calculating values based on decimal placement rules (e.g., comparing numbers by their non-zero decimal digits to detect **Decimal Place Value** errors).
* **Context-Specific Features:** Generating geometric relationships (e.g., **Exterior/Interior Angles** for geometry questions) and algebraic patterns (e.g., taking the last digit of an answer for **Equation** misconceptions).

---

### **Key Innovation: Feature Substitution** 💡

A crucial technique that made the models robust to varying numerical values was **substituting mathematical context features directly into the explanation text**. Instead of training on raw numerical values, the explanations were converted into templates with named placeholders.

**Example:**

**Original Question:**
> A container contains \( 60 \) tokens. The tokens are blue or yellow. \( \frac{2}{3} \) of the tokens are blue. How many blue tokens are there?

**Original Explanation:**
> "if 2 out of 3 are blue, then 60 times 2 divided by 3 equals 40"

**Feature-Substituted Explanation:**
> "if FRACTION_NUMERATOR out of FRACTION_DENOMINATOR are blue, then COMP_NUM_TIMES_WHOLE times FRACTION_NUMERATOR divided by FRACTION_DENOMINATOR equals NUM_TIMES_WHOLE_DIV_DEN_SIMP"

This substitution approach allows the models to:
* Learn **conceptual patterns** rather than memorizing specific numerical values
* Generalize to problems with different numbers
* Focus on the **mathematical relationships** being referenced in the explanation

The mathematical context features were appended to the text input, effectively giving the Qwen model a structured **numerical lexicon** to reason about the problem space.

---

### **Model Architecture & Performance**

The final system employed **two separate Qwen2-7B-Instruct Large Language Models**, each adapted using **QLoRA (Low-Rank Adaptation with 4-bit Quantization)**:

1. **Correctness Model:** Single-label classification to predict if the explanation is correct or incorrect
2. **Misconception Model:** Multi-label classification to identify specific misconception types

The combination of **separate specialized models** and **feature substitution** proved to be the key factors in achieving high performance and robustness to different numerical values.

The results on the validation set demonstrate strong performance across both tasks:

| Model | Task Type | F1 Score |
| :--- | :--- | :--- |
| **Correctness Model** | Single-label | **0.9822** |
| **Misconception Model** | Multi-label | **0.9835** |

The dual-model architecture, combined with data augmentation, feature substitution, and advanced feature engineering, resulted in high Macro F1 scores that confirm the system's ability to identify correct answers and accurately classify specific types of mathematical errors present in student reasoning—regardless of the specific numerical values used in the problems.

In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.

# Standard Library
import os
import sys
import re
import math
import string
import random
import warnings
import ast
from collections import Counter
from typing import Optional, List
from fractions import Fraction

# Third-party Data Processing
import numpy as np
import pandas as pd
import joblib

# Scikit-learn
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split as tts
from sklearn.metrics import f1_score, precision_recall_fscore_support

# Deep Learning and Transformers
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from torch.utils.data import DataLoader

# External Services and Environment
import kagglehub
from google.colab import drive
from google.colab import runtime





# Configuration
warnings.filterwarnings('ignore')


# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')
print("✅ Drive mounted successfully.")

# 2. Define the base directory for your project inside Drive
# 🔑 IMPORTANT: Change 'YourProject' to the actual folder name in your Drive
DRIVE_DIR = '/content/drive/MyDrive/'

# 3. Create the directory if it doesn't exist (good practice for saving checkpoints)
if not os.path.exists(DRIVE_DIR):
    os.makedirs(DRIVE_DIR)
    print(f"Created directory: {DRIVE_DIR}")
else:
    print(f"Using existing directory: {DRIVE_DIR}")
drive_dir=DRIVE_DIR
project_dir=os.path.join(drive_dir,'LLM_Misconception_Project')
checkpoint_dir=os.path.join(project_dir,'checkpoints')
scripts_dir=os.path.join(project_dir,'utils')
data_dir=os.path.join(project_dir,'data')


Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted successfully.
Using existing directory: /content/drive/MyDrive/


# **Get MAP Data from kagglehub**

In [2]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
try:
  map_charting_student_math_misunderstandings_path = kagglehub.competition_download('map-charting-student-math-misunderstandings')

  johnprichard_augment_function_map9_path = kagglehub.dataset_download('johnprichard/augment-function-map9')
except:

  map_charting_student_math_misunderstandings_path=os.path.join(project_dir,'data')

print('Data source import complete.')


Data source import complete.


# **Import Synthetic Data Generator**

In [3]:
# Assuming drive_dir = '/content/drive/MyDrive/YourProject' has been defined.

# 1. Construct the full path to the directory containing the module
module_directory = os.path.join(drive_dir, 'augment_functions_map38.py')

# 2. Add the DIRECTORY to the system path
# Note: You should add the *directory* that contains the file, not the file itself.
# In this case, since the file is expected to be *inside* the drive_dir,
# you should just append the drive_dir itself, as that's where Python will look.

if scripts_dir not in sys.path:
    sys.path.append(scripts_dir)
    print(f"✅ Added {scripts_dir} to sys.path.")
else:
    print(f"ℹ️ {scripts_dir} is already in sys.path.")

# 3. Now you can import the module directly
try:
    import augment_functions_map41 as augment_functions
    print("✅ Successfully imported augment_functions_map36.")
except ImportError as e:
    print(f"❌ Failed to import module: {e}")
    print("Ensure the file 'augment_functions_map41.py' is directly inside the directory specified by drive_dir.")
# Custom local modules
import preprocessing_functions as pf

✅ Added /content/drive/MyDrive/LLM_Misconception_Project/utils to sys.path.
Updated Module feb 15
✅ Successfully imported augment_functions_map36.


## 💾 **Data Augmentation Overview**

---

A total of **56,148 synthetic samples** were generated and added to the training set using the custom `augment_functions` module.

These augmented samples were created using **templates** derived from the original Kaggle data. This strategy ensures the generated questions and explanations maintain the same **linguistic structure and misconception type** but feature **randomized numerical values**, thereby drastically improving the model's ability to generalize to unseen numerical combinations.

In [4]:
# --- 1. Dependencies and Path Configuration ---
!pip install -U bitsandbytes accelerate peft
from bitsandbytes.optim import AdamW8bit
# NOTE: The augment functions are located inside the directory
# pointed to by johnprichard_augment_function_map9_path.
# We must add this path to the system path to allow Python to find the module.
# The path variable was created by the initial cell you ran.
try:
  sys.path.append(johnprichard_augment_function_map9_path)
except:
  print('Problem with kaggle server')




# --- 3. Initialize Flags and Load DataFrames ---
ANALYSIS = False
analysis = False

# Construct the full path to the CSV files using the competition path variable
# The files are typically located directly inside the downloaded competition directory.
train_path = os.path.join(map_charting_student_math_misunderstandings_path, 'train.csv')
test_path = os.path.join(map_charting_student_math_misunderstandings_path, 'test.csv')

try:
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)
    print('Competition data (train.csv and test.csv) loaded successfully.')
except FileNotFoundError as e:
    print(f"FATAL ERROR: Could not find data files at the expected path: {train_path}")
    print("Please verify the exact file names and structure within your Kaggle data source.")
    print(e)

question_id_map={31772:4,
                          31774:1,
                          31777:4,
                          31778:14,
                          32829:13,
                          32833: 3,
                          32835:7,
                          33471:4,
                          33472: 5,
                          33474: 9,
                          76870: 11,
                          89443: 12,
                          91695: 2,
                          104665: 8,
                          109465:10,
                 31772:15}
df_train['problem_type']=df_train['QuestionId'].apply(lambda x: question_id_map[x])


# --- 4. Data Cleanup and Augmentation ---
# Clean up the 'Wrong_fraction' inconsistency
#df_train['Misconception'] = df_train['Misconception'].apply(lambda x: 'Wrong_Fraction' if x == 'Wrong_fraction' else x)
df_train_copy = df_train.copy()

# Generate augmented data using your imported function
augmented_data_set = augment_functions.generate_all_augmented_data(change_cat_prob=0.20)
augmented_data_set['Original_category'] = augmented_data_set['Category'].apply(lambda x: x[0])


print('Initial data setup and augmentation complete. You are now running with the correct file references.')

Problem with kaggle server
Competition data (train.csv and test.csv) loaded successfully.
Number of augment functions called 60
Generated 18 examples for augment_adding_across1
Generated 840 examples for augment_subtraction
Generated 18 examples for augment_inverse_operation
Generated 3013 examples for augment_longer_is_bigger
Generated 120 examples for augment_multiplying_by_4
Generated 840 examples for augment_fraction_word_problems
Generated 2730 examples for augment_swap_dividend
Generated 4284 examples for augment_mult_misconception
Generated 1415 examples for augment_flipchange_misconception
Generated 373 examples for augment_inversion_misconception
Generated 8350 examples for augment_duplication_misconception
Generated 947 examples for augment_wrong_operation_misconception
Generated 1560 examples for augment_firstterm_misconception
Generated 1125 examples for augment_base_rate
Generated 3636 examples for augment_division
Generated 1385 examples for augment_scale_misconception
Ge

### **Ensure Consistent Answer Formatting**

In [5]:
# 1. Apply the standardization function to the 'MC_Answer' column
print("Applying standardization function...")
augmented_data_set['MC_Answer'] = augmented_data_set['MC_Answer'].apply(pf.standardize_answer_format_v5)

# 2. Get the unique list of *formatted* answers
unique_formatted_answers = augmented_data_set['MC_Answer'].unique()

# 3. Determine the number of samples to take (max 200 or the total unique count)
num_samples = min(200, len(unique_formatted_answers))

# 4. Randomly select the samples
print(f"Sampling {num_samples} unique formatted MC_Answers:")
sampled_answers = random.sample(list(unique_formatted_answers), num_samples)

# 5. Print the results
print("-" * 50)
for i, answer in enumerate(sampled_answers, 1):
    print(f"{i}. {answer}")
print("-" * 50)

Applying standardization function...
Sampling 200 unique formatted MC_Answers:
--------------------------------------------------
1. \( \frac{3}{6} \div \frac{1}{2} \)
2. \( 13.54 \)
3. \( \frac{124}{99} \)
4. \( \frac{4}{24} \)
5. \( 6.5285 \)
6. \( \frac{19}{88} \)
7. \( \frac{8}{56} \)
8. \( 686 \)
9. \( \frac{6}{54} \)
10. \( 83.89 \)
11. \( 10 \frac{6}{11} \)
12. \( 5.03132 \)
13. \( 43.028 \)
14. \( 1400 \) seconds
15. \( \frac{135}{110} \)
16. \(0\)
17. \( 16.481 \)
18. \( 9.0 \) minutes
19. \( 10.6724 \)
20. \( \frac{15}{3}\)
21. \(2.000081\)
22. \( \frac{37}{48} \)
23. \( 41.02 \)
24. \( 21 \) days
25. \( 6.38 \)
26. \( 22.3066 \)
27. \( \frac{1}{3} \div \frac{1}{8} \)
28. \( 2016 \)
29. \( 24.3 \)
30. \( \frac{1}{8} \times \frac{2}{4} \)
31. \( 192 \) minutes
32. \( 63.38 \)
33. \( 16.13 \)
34. \( \frac{75}{112} \)
35. \( 21.8 \)
36. \( -42 \)
37. \( \frac{25}{2} \)
38. \( 9.434 \)
39. \( \frac{2}{4} \times \frac{1}{3} \)
40. \( 2.000003 \)
41. \( 18 \frac{1}{10} \)
42. \( \f

In [6]:
# Track labels from synthetic data in order to ensure consistency with kaggle data.
all_augmented_labels=set()
for ind, row in augmented_data_set.iterrows():
    mis=row['Misconception']
    for label in mis:
        all_augmented_labels.add(label)
print('Number of Labels in Augmented Data: ', len(all_augmented_labels))
print(all_augmented_labels)


Number of Labels in Augmented Data:  37
{'Longer_is_bigger', 'Wrong_fraction', 'Additive', 'Incomplete', 'Adding_terms', 'Irrelevant', 'Adding_across', 'Denominator-only_change', 'Shorter_is_bigger', 'Firstterm', 'Base_rate', 'Not_variable', 'Wrong_Operation', 'Inverse_operation', 'SwapDividend', 'Division', 'Duplication', 'Wrong_Fraction', 'Certainty', 'Incorrect_equivalent_fraction_addition', 'Interior', 'Mult', 'Definition', 'Other', 'Subtraction', 'No Misconception', 'Multiplying_by_4', 'Inversion', 'Positive', 'Unknowable', 'Wrong_term', 'FlipChange', 'Tacking', 'Ignores_zeroes', 'Whole_numbers_larger', 'WNB', 'Scale'}


# ⚙️ Data Preprocessing: Preparing for Multi-Label Classification

This section processes both the original Kaggle data (`df_train`) and the augmented data (`augmented_data_set`) to prepare the target columns for multi-label training.

### **1. Target Column Transformation**

The `process_df_for_multi_label` function is used to aggregate the original data by unique `(QuestionText, StudentExplanation)` pairs. This ensures that if a single explanation was tagged with multiple different categories (which happens when an error fits several distinct labels), the categories are grouped into a single list.

* **Goal:** Convert the `Category` and `Misconception` columns from single-string entries into **lists** (e.g., `'True_A'` becomes `['True_A']`), which is required for multi-label classification using the model heads.
* **Multi-Label Creation:** The grouping logic explicitly allows for the creation of multi-label samples where one student explanation is associated with multiple `Category` values.

### **2. Correctness Column Creation**

The binary target column, `Correctness`, is derived directly from the `Category` string for both datasets:

* **`Correctness` (Binary Target):**
    * **1 (True):** If the `Category` starts with `'True'` (i.e., `True_Correct`, `True_Misconception`, `True_Neither`).
    * **0 (False):** If the `Category` starts with `'False'` (i.e., `False_Correct`, `False_Misconception`, `False_Neither`).

### **3. Category Definitions (Correctness & Misconception Status)**

The `Category` column serves as a combined label, simultaneously specifying the correctness of the final answer and the general presence of an identified misconception.

| Category | Correctness | Misconception Status | Description |
| :--- | :--- | :--- | :--- |
| **`True_Correct`** | **Correct** | **Absent** | The final answer is correct, and no specific misconception is present in the explanation. |
| **`True_Misconception`** | **Correct** | **Present** | The final answer is correct, but a specific misconception from the 35 available classes is present in the explanation. |
| **`False_Correct`** | **Incorrect** | **Absent** | The final answer is incorrect, but no specific misconception is present (perhaps a calculation error). |
| **`False_Misconception`** | **Incorrect** | **Present** | The final answer is incorrect, and a specific misconception from the 35 available classes is present. |
| **Special Categories** | | | |
| **`True_Neither`** | **Correct** | **Uncategorized** | A specific error exists, but it does not fit into any of the 35 defined misconception categories. |
| **`False_Neither`** | **Incorrect** | **Uncategorized** | A specific error exists, but it does not fit into any of the 35 defined misconception categories. |

In [7]:
def process_df_for_multi_label(df):
    duplicate_questions = df_train[df_train.duplicated(subset=['StudentExplanation', 'QuestionText','Category'], keep=False)]
    duplicated_rows=duplicate_questions['row_id'].values

    possible_multi_label_rows=df[~df['row_id'].isin(duplicated_rows)]
    grouped = possible_multi_label_rows.groupby(['QuestionText', 'StudentExplanation']).agg({
        'row_id':'first',
        'QuestionId':'first',
        'Category': list,  # aggregate categories as list
        'MC_Answer': 'first',
        'Misconception':'first',
        'Correctness':'first',
        'problem_type':'first'

    }).reset_index()
    duplicate_questions['Category']=duplicate_questions['Category'].apply(lambda x: [x])
    grouped=grouped[['row_id', 'QuestionId', 'QuestionText', 'MC_Answer',
       'StudentExplanation', 'Category', 'Misconception','Correctness','problem_type']]
    duplicate_questions=duplicate_questions[['row_id', 'QuestionId', 'QuestionText', 'MC_Answer',
       'StudentExplanation', 'Category', 'Misconception','Correctness','problem_type']]
    final_df=pd.concat([grouped,duplicate_questions])
    return final_df

df_train['Correctness'] = df_train['Category'].apply(lambda x: 1 if x.split('_')[0] == 'True' else 0)

augmented_data_set['Correctness']=augmented_data_set['Original_category'].apply(lambda x: 1 if x.split('_')[0] == 'True' else 0)
df_train=process_df_for_multi_label(df_train)
df_train.head()

,row_id,QuestionId,QuestionText,MC_Answer,StudentExplanation,Category,Misconception,Correctness,problem_type
0,22984,33471,A bag contains \( 24 \) yellow and green balls...,\( 9 \),(using the explanation above) 24 divided by fo...,[False_Neither],None,0,4
1,22014,33471,A bag contains \( 24 \) yellow and green balls...,\( 15 \),.\n24- 8=3x3=9 and i thought it was higher i ...,[True_Neither],None,1,4
2,22888,33471,A bag contains \( 24 \) yellow and green balls...,\( 3 \),1 / 8 = 3 and you want to find out what 5/8 is...,[False_Correct],None,0,4
3,22985,33471,A bag contains \( 24 \) yellow and green balls...,\( 9 \),1 / 8 = 3 so i dud 3 x 3 to get 9.,[False_Misconception],Wrong_fraction,0,4
4,22986,33471,A bag contains \( 24 \) yellow and green balls...,\( 9 \),1 / 8 of 24 is 3 so three times 3 is 9.,[False_Misconception],Wrong_fraction,0,4


In [8]:
# Create a combined stratify column by concatenation
df_train['stratify_col'] = df_train['Misconception'].astype(str) + "_" + df_train['Category'].astype(str)

counts = df_train['stratify_col'].value_counts()
rare_strata = counts[counts < 7].index
print(rare_strata)

df_train.loc[df_train['stratify_col'].isin(rare_strata), 'stratify_col'] = 'other'


# Then run train_test_split using this combined column for stratify
train, test = tts(df_train, stratify=df_train['stratify_col'], test_size=0.3, random_state=5)
test, val = tts(test, stratify=test['stratify_col'], test_size=0.4, random_state=5)

# Optional: drop the combined column if no longer needed
df_train.drop(columns=['stratify_col'], inplace=True)

number_of_classes = len(df_train['Misconception'].unique())
print('Number of Classes: ', number_of_classes)
print(f"Training Samples: {len(train)}")
print(f"Validation Samples: {len(val)}")
print(f"Testing Samples: {len(test)}")


train_tuples = list(zip(train['QuestionText'], train['StudentExplanation'], train['MC_Answer']))
test_tuples = list(zip(test['QuestionText'], test['StudentExplanation'], test['MC_Answer']))
val_tuples = list(zip(val['QuestionText'], val['StudentExplanation'], val['MC_Answer']))





Index(['Shorter_is_bigger_['False_Misconception']',
       'Wrong_fraction_['True_Misconception']',
       'Incomplete_['True_Misconception']',
       'Wrong_Operation_['False_Misconception']',
       'Duplication_['True_Misconception']', 'Division_['True_Misconception']',
       'Inversion_['True_Misconception']',
       'None_['True_Neither', 'True_Correct']',
       'Denominator-only_change_['True_Misconception']',
       'FlipChange_['True_Misconception']',
       'None_['False_Neither', 'True_Correct']',
       'Incomplete_['False_Neither', 'False_Misconception']',
       'Definition_['True_Misconception']',
       'Multiplying_by_4_['True_Misconception']',
       'None_['False_Neither', 'False_Correct']',
       'Wrong_term_['False_Neither', 'False_Misconception']',
       'Subtraction_['True_Misconception']',
       'Incorrect_equivalent_fraction_addition_['True_Misconception']',
       'Positive_['True_Misconception']',
       'Positive_['False_Misconception', 'False_Neither']'

## **Label Encoders for Category**


In [9]:
def create_label_encoder(unique_categories):
    le=LabelEncoder()
    le.fit(unique_categories)
    return le
unique_cats=df_train_copy['Category'].unique()
label_encoder_category=create_label_encoder(unique_cats)


# **Misconception Label Encoder**

In [10]:
def fill_na(row, fill_value='No Misconception'):
    # Check if the value is NaN
    if pd.isna(row['Misconception']):
        if row['Category'].split('_')[1]=='Correct':
            return 'No Misconception'
        else:
            return 'Other'
    else:
        return row['Misconception']
new_df_train=df_train_copy.copy()
new_df_train['Misconception']=new_df_train.apply(fill_na, axis=1)

unique_misconceptions=new_df_train['Misconception'].unique()
label_encoder_misconception=create_label_encoder(unique_misconceptions)
num_categories=len(label_encoder_category.classes_)
num_misc_classes=len(label_encoder_misconception.classes_)
print(f"Number of Category Classes: {num_categories}")
print(f"Number of Misconception Classes: {num_misc_classes}")

SAVE_LE=False
if SAVE_LE:
    joblib.dump(label_encoder_category, os.path.join(project_dir,'label_encoder_category.joblib'))
    joblib.dump(label_encoder_misconception, os.path.join(project_dir,'label_encoder_misconception.joblib'))

Number of Category Classes: 6
Number of Misconception Classes: 37


# **Splitting Original Kaggle Data**


In [11]:
def aggregate_by_question_answer(df):
    grouped = df.groupby(['QuestionText', 'StudentExplanation', 'MC_Answer']).agg({
        'Category': lambda x: list(set(x)),           # unique categories per group
        'Misconception': lambda x: list(set(x)),      # unique misconceptions per group
        'Correctness': 'first',                        # correctness label per group - assumes consistent
        'row_id': lambda x: list(x),
        'problem_type':'first'# list of all row_ids in the group
    }).reset_index()
    return grouped

use_relabeled_other_df=True
# Example usage:
new_df_train['Correctness'] = new_df_train['Category'].apply(lambda x: 1 if x.split('_')[0] == 'True' else 0)
new_df_train_copy=new_df_train.copy()
other_df=new_df_train_copy[new_df_train_copy['Misconception']=='Other']
new_df_train=new_df_train[new_df_train['Misconception']!='Other']

other_df_agg=aggregate_by_question_answer(other_df)
agg_df=aggregate_by_question_answer(new_df_train)
agg_df['tuple_key'] = list(zip(agg_df['QuestionText'], agg_df['StudentExplanation'], agg_df['MC_Answer']))
other_df_agg['tuple_key'] = list(zip(other_df_agg['QuestionText'], other_df_agg['StudentExplanation'], other_df_agg['MC_Answer']))
agg_train = agg_df[agg_df['tuple_key'].isin(train_tuples)]

# Overwrite other_df_agg with relabeled/cleaned data
if use_relabeled_other_df:
  other_df_agg=pd.read_csv(os.path.join(data_dir,'other_samples_relabeled.csv'))
other_df_agg_val = other_df_agg[other_df_agg['tuple_key'].isin(val_tuples)]
other_df_agg_train = other_df_agg[other_df_agg['tuple_key'].isin(train_tuples)]
other_df_agg_test=other_df_agg[other_df_agg['tuple_key'].isin(test_tuples)]
agg_test = agg_df[agg_df['tuple_key'].isin(test_tuples)]
agg_val = agg_df[agg_df['tuple_key'].isin(val_tuples)]
agg_val_copy=agg_val.copy()

extras=agg_df[(~agg_df['tuple_key'].isin(train_tuples))&(~agg_df['tuple_key'].isin(val_tuples))\
&(~agg_df['tuple_key'].isin(test_tuples))]
print(len(extras))
agg_train=pd.concat([agg_train,extras])



7


## 🧪 **Strategy: Augmenting the "Neither" Category**

The "Neither" category is crucial for training the model to recognize explanations that are **not based on specific mathematical reasoning** (correct or incorrect) but rather represent non-explanations, random guesses, or attempts that cannot be classified by the existing 35 misconception types. Since the custom data augmentation focused on generating specific misconception types, the `_Neither` category was inherently sparse.

### **Implementation Details**

To address this sparsity and explicitly teach the model the pattern of non-explanation, the `create_neither_samples` function was used:

1.  **Source Selection:** The function sampled 30% of existing **Misconception** entries (both `True_Misconception` and `False_Misconception`) from the training data.
2.  **Explanation Replacement:** The original student explanation was replaced with a **synthesized "vague" explanation**. This involved substituting the original text with strings representing:
    * **Empty/Blank input** (weighted heavily).
    * **Random Guesses** ("I just guessed," "I don't know").
    * **Letter Preferences** ("I just like the letter A").
    * **Vague Intuition** ("It just seems right").
3.  **Label Switching:** For the sampled entries, the labels were forcefully updated:
    * **Category:** The category suffix was changed from `_Misconception` to **`_Neither`** (e.g., `True_Misconception` became `True_Neither`).
    * **Misconception:** The multi-label list was set to **`['Other']`**, signaling to the misconception task head that no specific error from the 35 classes was present.

### **Impact on Model Training**

This targeted augmentation serves two primary purposes:

* **Robustness to Noise:** It directly trains the model to classify noisy, non-explanatory, or blank student input into the `_Neither` category, preventing these inputs from being misclassified as specific, reasoned misconceptions.
* **Encouraging Explanatory Reliance:** By mapping high-quality, reasoned explanations (the originals) to generic, uninformative explanations while preserving the answer's correctness, this process forces the model to **heavily consider the quality and content of the `StudentExplanation`** for its final classification. If the explanation is vague, the model is penalized for predicting a specific misconception.

In [12]:
"""
Create additional training samples for the "Neither" category.
These samples teach the model to recognize non-explanations, random guesses, etc.
"""

def create_neither_samples(df, sample_fraction=0.3, seed=42):
    """
    Create "Neither" category samples from existing misconception samples.

    Args:
        df: DataFrame with columns ['QuestionText', 'StudentExplanation', 'Category',
                                     'Misconception', 'MC_Answer', 'Correctness', ...]
        sample_fraction: Fraction of misconception samples to convert (default 0.3 = 30%)
        seed: Random seed for reproducibility

    Returns:
        DataFrame with new "Neither" samples
    """

    random.seed(seed)
    np.random.seed(seed)

    # Filter for misconception samples (False_Misconception or True_Misconception)
    misconception_mask = df['Category'].apply(
        lambda x: isinstance(x, list) and any('Misconception' in cat for cat in x)
    )
    misconception_samples = df[misconception_mask].copy()

    # Sample the specified fraction
    n_samples = int(len(misconception_samples) * sample_fraction)
    sampled_indices = misconception_samples.sample(n=n_samples, random_state=seed).index

    print(f"\n📊 Creating 'Neither' samples:")
    print(f"  Total misconception samples: {len(misconception_samples)}")
    print(f"  Converting {n_samples} samples ({sample_fraction*100:.1f}%)")

    # Create new DataFrame for neither samples
    neither_samples = df.loc[sampled_indices].copy()

    # Define replacement explanations
    empty_explanations = [
        "",  # Empty string
        " ",  # Single space
        "  ",  # Multiple spaces
    ]

    random_guess_explanations = [
        "I don't know I just chose this answer",
        "I just guessed",
        "I picked this one randomly",
        "I'm not sure why I chose this",
        "No idea, just picked one",
        "Random guess",
        "I don't know",
        "idk",
        "Just guessing",
        "I chose randomly",
        "Not sure",
        "I just picked something",
        "I guessed this one",
        "Random choice",
        "I don't really know"
    ]

    letter_preference_explanations = [
        "I just like the letter A",
        "I just like the letter B",
        "I just like the letter C",
        "I just like the letter D",
        "I always pick A",
        "I always pick C",
        "B is my lucky letter",
        "I prefer option A",
        "C looked good to me",
        "I like the third option",
        "First one looked good",
        "Last one seemed right"
    ]

    vague_explanations = [
        "It just seems right",
        "This one looks correct",
        "I think this is it",
        "Feels like the right answer",
        "This seems like the answer",
        "I feel like this is correct",
        "Just my intuition",
        "It looked right to me",
        "This one stood out"
    ]

    # Combine all explanation types
    all_replacement_explanations = (
        empty_explanations * 20 +  # Weight empty explanations more
        random_guess_explanations * 10 +
        letter_preference_explanations * 5 +
        vague_explanations * 5
    )

    # Randomly assign replacement explanations
    neither_samples['StudentExplanation'] = [
        random.choice(all_replacement_explanations)
        for _ in range(len(neither_samples))
    ]

    # Update Category: Change second part after '_' to 'Neither'
    def update_category(cat_list):
        """Change 'Misconception' or 'Correct' to 'Neither' in category."""
        if not isinstance(cat_list, list):
            cat_list = [cat_list]

        updated_cats = []
        for cat in cat_list:
            if '_' in cat:
                parts = cat.split('_')
                # Keep first part (True/False), change second part to 'Neither'
                updated_cats.append(f"{parts[0]}_Neither")
            else:
                updated_cats.append(cat)

        return updated_cats

    neither_samples['Category'] = neither_samples['Category'].apply(update_category)

    # Update Misconception to 'Other'
    neither_samples['Misconception'] = [['Other'] for _ in range(len(neither_samples))]

    # Count by explanation type for reporting
    empty_count = sum(1 for exp in neither_samples['StudentExplanation'] if exp.strip() == '')
    guess_count = sum(1 for exp in neither_samples['StudentExplanation']
                     if any(word in exp.lower() for word in ['guess', "don't know", 'idk', 'random']))
    letter_count = sum(1 for exp in neither_samples['StudentExplanation']
                      if 'letter' in exp.lower() or 'option' in exp.lower())
    vague_count = len(neither_samples) - empty_count - guess_count - letter_count

    print(f"\n📝 Explanation type distribution:")
    print(f"  Empty/blank: {empty_count} ({empty_count/len(neither_samples)*100:.1f}%)")
    print(f"  Random guess: {guess_count} ({guess_count/len(neither_samples)*100:.1f}%)")
    print(f"  Letter preference: {letter_count} ({letter_count/len(neither_samples)*100:.1f}%)")
    print(f"  Vague: {vague_count} ({vague_count/len(neither_samples)*100:.1f}%)")

    # Count category distribution
    category_counts = neither_samples['Category'].apply(lambda x: x[0]).value_counts()
    print(f"\n📊 Category distribution:")
    for cat, count in category_counts.items():
        print(f"  {cat}: {count}")

    return neither_samples


def combine_with_original(original_df, neither_samples):
    """
    Combine original dataset with new 'Neither' samples.

    Args:
        original_df: Original training DataFrame
        neither_samples: DataFrame with 'Neither' samples

    Returns:
        Combined DataFrame
    """
    # Concatenate
    combined_df = pd.concat([original_df, neither_samples], ignore_index=True)

    # Shuffle
    combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"\n✅ Combined dataset:")
    print(f"  Original samples: {len(original_df)}")
    print(f"  Neither samples: {len(neither_samples)}")
    print(f"  Total samples: {len(combined_df)}")

    return combined_df


# ============================================================================
# USAGE EXAMPLE
# ============================================================================



print("\n" + "🎯 "*40)
print("CREATING 'NEITHER' CATEGORY SAMPLES")
print("🎯 "*40)

# Assuming you have 'augmented_data_set' or 'final_training_set'
# Replace with your actual DataFrame variable name
# The correct way to filter out specific single-item lists from a list column
df_filtered = augmented_data_set[
    (augmented_data_set['Misconception'].apply(lambda x: 'No Misconception' not in x)) &
    (augmented_data_set['Misconception'].apply(lambda x: 'Other' not in x))
]
# Example: Create Neither samples from 30% of misconception samples
neither_samples = create_neither_samples(
    df=df_filtered,  # or augmented_data_set
    sample_fraction=0.3,     # Convert 30% of misconception samples
    seed=42
)
neither_samples['Correctness']=neither_samples['Original_category'].apply(lambda x: 1 if x.split('_')[0] == 'True' else 0)





🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 
CREATING 'NEITHER' CATEGORY SAMPLES
🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 

📊 Creating 'Neither' samples:
  Total misconception samples: 60193
  Converting 18057 samples (30.0%)

📝 Explanation type distribution:
  Empty/blank: 3402 (18.8%)
  Random guess: 5732 (31.7%)
  Letter preference: 2015 (11.2%)
  Vague: 6908 (38.3%)

📊 Category distribution:
  False_Neither: 15454
  True_Neither: 2575
  TRUE_Neither: 28


## ✂️ Data Splitting: Stratified Sampling

The dataset was split into training, validation, and testing sets using **stratified sampling** to ensure a balanced distribution of critical target labels across all three partitions.

### **Method**

1.  **Combined Stratification:** A composite column (`stratify_col`) was created by concatenating the **`Misconception`** and **`Category`** labels. This ensures that the proportion of every unique combination of misconception type and category (e.g., `['Inversion']_['False_Misconception']`) is maintained in the final splits.
2.  **Handling Rare Strata:** Any strata (unique label combinations) with fewer than 7 samples were grouped into an **`'other'`** class to prevent splitting errors during the `train_test_split` operation.
3.  **Splitting:** The data was split with the following ratios:
    * **70%** for the initial training set (`train`).
    * The remaining **30%** was split into a **Validation Set (12%)** and a **Testing Set (18%)**.

This process guarantees that the resulting three datasets (`train`, `val`, `test`) have a representative and proportionate count of all major classification targets.

* **Training Samples:** 79,906
* **Validation Samples:** 13,700
* **Testing Samples:** 20,551
* **Total Misconception Classes:** 37

## 🔄 Synthetic Data Split Mechanism

The line of code `augmented_data_set_val=augment_functions.generate_all_augmented_data(which='val')` utilizes a built-in mechanism within your `augment_functions` module to manage the split between synthetic training and synthetic validation data.

This is a **critical safeguard** against data leakage:

* **Template Separation:** The augmentation system is designed to use **different sets of templates** for generating the training data versus the validation data.
* **Unique Explanations:** When you pass the argument `which='val'`, the function generates synthetic samples using templates and/or random seeds that produce **unique student explanations and numerical combinations** that were **not used** in the main synthetic training set.

This ensures that the model is validated on genuinely unseen numerical variations and explanation structures, preventing your performance metrics (like the high F1 scores) from being artificially inflated by testing on data too similar to what was used during training.

In [13]:
augmented_data_set_val=augment_functions.generate_all_augmented_data(which='val',change_cat_prob=0)
augmented_data_set_val['MC_Answer'] = augmented_data_set_val['MC_Answer'].apply(pf.standardize_answer_format_v5)


print('Augmented Validation set Size: ', len(augmented_data_set_val))
# --- STEP 1: DEFINE RARE CLASSES AND FLATTEN THE MISCONCEPTION LISTS ---

# 1a. Define the rare classes (those with counts < 100 based on your output)
RARE_THRESHOLD = 100
rare_misconceptions = [
    '[Positive, Adding_across]',
    '[Incomplete, SwapDividend]',
    '[Incomplete, WNB]',
    '[WNB, Wrong_Fraction]',
    '[Incomplete, Subtraction]',
    '[Firstterm, Inversion]',
    # Include the multi-misconception strings that were below the threshold
    # Note: These are likely lists in your data, so we'll flatten everything first.
]

# 1b. Function to flatten the list in the 'Misconception' column to a clean string
def flatten_misconception(item):
    """Converts a list of misconceptions (e.g., ['Scale']) to a string (e.g., 'Scale')."""
    if isinstance(item, list) and len(item) > 0:
        # We join to handle the few cases with multiple misconceptions
        return " | ".join(item)
    elif isinstance(item, list) and len(item) == 0:
        return "No Misconception"
    else:
        # Remove brackets from single-item lists that may have already been converted to strings
        return str(item).strip('[]')

# Apply the flattening function to the Misconception column
augmented_data_set_val['Misconception_Clean'] = augmented_data_set_val['Misconception'].apply(flatten_misconception)


# 1c. Recalculate counts on the clean column to identify all items below the threshold
class_counts_clean = augmented_data_set_val['Misconception_Clean'].value_counts()
rare_misconceptions_to_group = class_counts_clean[class_counts_clean < RARE_THRESHOLD].index.tolist()


# --- STEP 2: GROUP RARE CLASSES ---

# Create the stratification column by grouping all rare classes
augmented_data_set_val['Misconception_Stratify'] = augmented_data_set_val['Misconception_Clean'].apply(
    lambda x: 'Z_RARE_GROUP' if x in rare_misconceptions_to_group else x
)


# --- STEP 3: PERFORM THE STRATIFIED SPLIT ---

# Assuming 'tts' is your imported train_test_split function
train_df, val_df = tts(
    augmented_data_set_val,
    test_size=0.6,
    # Stratify on the newly created column that has no single-count classes
    stratify=augmented_data_set_val['Misconception_Stratify'],
    random_state=42 # Good practice to keep the split consistent
)

# You can now use the 'val_df' which is a DataFrame containing your validation set.
print('✅ Preprocessing Complete.')
print('Augmented Validation set Size: ', len(val_df))
print('\nMisconception Counts for Stratification Column:')
print(augmented_data_set_val['Misconception_Stratify'].value_counts())

augmented_data_set_val['Original_category'] = augmented_data_set_val['Category'].apply(lambda x: x[0])
augmented_data_set_val['Correctness']=augmented_data_set_val['Original_category'].apply(lambda x: 1 if x.split('_')[0] == 'True' else 0)

Number of augment functions called 57
Generated 17 examples for augment_adding_across1
Generated 840 examples for augment_subtraction
Generated 18 examples for augment_inverse_operation
Generated 3007 examples for augment_longer_is_bigger
Generated 120 examples for augment_multiplying_by_4
Generated 760 examples for augment_fraction_word_problems
Generated 2730 examples for augment_swap_dividend
Generated 4284 examples for augment_mult_misconception
Generated 1439 examples for augment_flipchange_misconception
Generated 264 examples for augment_inversion_misconception
Generated 7058 examples for augment_duplication_misconception
Generated 685 examples for augment_wrong_operation_misconception
Generated 1560 examples for augment_firstterm_misconception
Generated 1125 examples for augment_base_rate
Generated 3600 examples for augment_division
Generated 1387 examples for augment_scale_misconception
Generated 1017 examples for augment_definition
Generated 1520 examples for augment_wnb
Gener

# Ensure Matching Class Names between synthetic and kaggle data

In [14]:
check=list(all_augmented_labels)
found=False
for lab in check:
    if lab not in label_encoder_misconception.classes_:
        print(lab)
        found=True

if not found:
    print('All labels in augmented data are present in original data')
found=False
for lab in label_encoder_misconception.classes_:
    if lab not in check:
        print(lab)
if not found:

    print('All labels in original data are present in augmented data')

All labels in augmented data are present in original data
All labels in original data are present in augmented data


## 🔄 Category Augmentation: Creating `False_Correct` Samples

This code block implements a custom augmentation strategy designed to specifically balance the **`Correct`** and **`Incorrect`** answer categories by generating synthetic `False_Correct` samples.

The core goal is to teach the model that **a correct explanation does not always guarantee a correct final answer**. The model must learn to classify the quality of the explanation independently of the provided answer.

### **Implementation (`CategoryAugmenter` Class)**

1.  **Selection (`correctness_split`):** The process starts by filtering the data to select only samples that originally had a **Correct Answer (`Correctness=1`)** AND **No Misconception** (`Misconception='No Misconception'`). These are the samples with *correct reasoning* and a *correct answer*.
2.  **Answer Distortion (`create_distractor_answer`):** The key manipulation occurs here. It takes the original correct answer (e.g., `\(\frac{6}{2}\)`) and generates a **distractor answer** by slightly modifying one number within the answer string (e.g., changes it to `\(\frac{7}{2}\)`).
3.  **Label Flip (`augment_correct_data`):**
    * The `MC_Answer` is replaced with the newly generated distractor answer.
    * The binary `Correctness` label is flipped from `1` (True) to **`0` (False)**.
    * The full `Category` string is updated (e.g., `True_Correct` becomes **`False_Correct`**).

### **Result**

The final `changed_cats` DataFrame contains new samples where the **`StudentExplanation` and underlying logic are correct (or contain No Misconception), but the final `MC_Answer` is intentionally wrong.**

This augmentation forces the model to learn that a high-quality explanation can be paired with an incorrect answer (e.g., due to a simple calculation or copying error). This helps the model generalize across the `Category` types by ensuring the Misconception/No Misconception classification is based on the **explanation content**, not just the provided answer.

The final samples are taken from only 50% of the generated data (`.sample(frac=0.5)`) to manage the final dataset size.

In [15]:
class CategoryAugmenter:
    def __init__(self,df):
        self.df=df
        self.correct_df,self.incorrect_df=self.correctness_split(self.df)

    def augment_correct_data(self):
        self.correct_df['MC_Answer']=self.correct_df['MC_Answer'].apply(self.create_distractor_answer)
        self.correct_df['Category']=self.correct_df['Category'].apply(self.change_category)
        self.correct_df['Correctness'] = 0
        return self.correct_df
    def change_category(self,category):
        new_category=[]
        for cat in category:
            misc=cat.split('_')[1]
            if cat.split('_')[0]=='True':
                new_category.append('False'+'_'+misc)
            else:
                new_category.append('True'+'_'+misc)

        return new_category

    def correctness_split(self,df):
        correct_df=df[df['Correctness']==1]
        def is_misconception(misc):
            found=False
            for item in misc:
                if item=='No Misconception':
                    return True
            return found
        correct_df['hasMisconception']=correct_df['Misconception'].apply(is_misconception)
        correct_no_misc=correct_df[correct_df['hasMisconception']]
        return correct_no_misc, df[df['Correctness']==0]

    def create_distractor_answer(self,answer_str: str) -> str:
        """
        Takes a LaTeX math answer and creates a distractor by replacing
        one number with a similar but different number.

        Examples:
            "\\( \\frac{6}{2} \\)" -> "\\( \\frac{7}{2} \\)" or "\\( \\frac{6}{3} \\)"
            "\\( 3 \\)" -> "\\( 4 \\)" or "\\( 2 \\)"
            "\\( \\frac{2}{3} \\div \\frac{1}{3} \\)" -> variations with changed numbers
        """

        def get_similar_number(num_str: str) -> str:
            """Generate a similar but different number"""
            try:
                num = int(num_str)

                # For small numbers (1-3), add or subtract 1
                if num <= 3:
                    options = [max(1, num - 1), num + 1]
                    options = [n for n in options if n != num]
                    return str(random.choice(options))

                # For larger numbers, use multiple strategies
                strategies = []

                # Strategy 1: Add/subtract 1
                strategies.extend([num - 1, num + 1])

                # Strategy 2: Add/subtract a small amount (2-3)
                if num > 5:
                    strategies.extend([num - 2, num + 2])

                # Strategy 3: Nearby number
                if num > 10:
                    strategies.extend([num - 3, num + 3])

                # Filter out the original number and negatives
                strategies = [n for n in strategies if n != num and n > 0]

                return str(random.choice(strategies))

            except ValueError:
                # If it's a decimal
                try:
                    num = float(num_str)
                    change = random.choice([-1, 1]) * random.uniform(0.5, 2.0)
                    new_num = max(0.1, num + change)
                    return f"{new_num:.2f}".rstrip('0').rstrip('.')
                except:
                    return num_str

        # Find all numbers in the string (integers and decimals)
        number_pattern = r'\d+\.?\d*'
        numbers = re.findall(number_pattern, answer_str)

        if not numbers:
            return answer_str  # No numbers found, return original

        # Pick a random number to replace
        num_to_replace = random.choice(numbers)
        new_num = get_similar_number(num_to_replace)

        # Replace only the first occurrence to avoid changing all instances
        # Use a more precise replacement that maintains the exact match
        result = answer_str.replace(num_to_replace, new_num, 1)

        return result


data_for_cat_aug=augmented_data_set[(augmented_data_set['problem_type']!=10)&(augmented_data_set['Misconception']!='Unknowable')]
changed_cats=CategoryAugmenter(data_for_cat_aug).augment_correct_data()
changed_cats=changed_cats.sample(frac=0.5)

## 🧹 Text Preprocessing: Standardizing Student Explanations for Feature Alignment 🔗

This code block initializes the `TextAugmenter` class and applies its essential **cleaning** function to standardize the `StudentExplanation` column across all data subsets.

### **Purpose: Bridging Text and Features**

The primary and most critical goal of this cleaning step is to **create a consistent numerical vocabulary** in the student explanations that directly matches the format used by the **Mathematical Context Features** (generated in a previous step).

* **The Problem:** If a student writes "I took **one half** of six," and the mathematical features for the question are labeled with `FRAC:1/2`, the model struggles to connect "one half" to "1/2".
* **The Solution:** The `clean()` method ensures all numerical and fractional expressions within the text are converted to their standard numeric format. This forces a direct alignment between the text and the engineered features.

### **Cleaning Transformations**

The `TextAugmenter.clean()` method performs the following transformations in a specific, crucial order:

1.  **Word Fraction to Number (`word_fraction_to_number`):** Converts complex fractional phrases written out as words into the standard numeric fraction format.
    * **Example:** "two thirds" $\rightarrow$ **"2/3"**
    * **Result:** The model can now associate the feature `FRAC:2/3` with the words "2/3" in the explanation.
2.  **Word to Number (`word_to_number`):** Converts simple whole number words to their digit equivalent.
    * **Example:** "four" $\rightarrow$ **"4"**
3.  **Normalize Fraction Text (`normalize_fraction_text`):** Converts verbose fraction representations using the word "over" into the standard division format.
    * **Example:** "4 over 6" $\rightarrow$ **"4/6"**

By standardizing the text, you provide the LLM with a unified input that allows it to effectively use the **mathematical features** to interpret the numerical reasoning (or lack thereof) in the student's response.

In [16]:
# --- Example Usage (Confirming the Fix) ---
augmenter = pf.TextAugmenter(augmentation_prob=1.0)

# New Test Case for the Cleaning Order
test_cleaning_order = "The fraction is four over eight."
print(f"Original Text: {test_cleaning_order}")
cleaned_result = augmenter.clean(test_cleaning_order)
print(f"Cleaned Result: {cleaned_result}\n")
# Expected Output: "The fraction is 4/8."

# Re-testing the original broken case
original_text = "Original Text for Cleaning: I think the answer is four over six because I multiply three times one."
print(f"Original Text: {original_text}")
cleaned_text = augmenter.clean("I think the answer is four over six because I multiply three times one.")
print(f"Cleaned Result: {cleaned_text}")
# Expected Output: "I think the answer is 4/6 because I multiply 3 times 1."


agg_val['StudentExplanation'] = agg_val['StudentExplanation'].apply(augmenter.clean)
other_df_agg['StudentExplanation'] = other_df_agg['StudentExplanation'].apply(augmenter.clean)
other_df_agg_train['StudentExplanation'] = other_df_agg_train['StudentExplanation'].apply(augmenter.clean)
other_df_agg_val['StudentExplanation'] = other_df_agg_val['StudentExplanation'].apply(augmenter.clean)

agg_test['StudentExplanation'] = agg_test['StudentExplanation'].apply(augmenter.clean)
other_df_agg_test['StudentExplanation'] = other_df_agg_test['StudentExplanation'].apply(augmenter.clean)
augmenter.clean(' two ')

Original Text: The fraction is four over eight.
Cleaned Result: The fraction is 4/8.

Original Text: Original Text for Cleaning: I think the answer is four over six because I multiply three times one.
Cleaned Result: I think the answer is 4/6 because I multiply 3 times one.


'2'

In [17]:
# Preprocessing steps for validation set
agg_val['QuestionText']=agg_val.apply(lambda x: pf.rephrase_question_and_explanation(x['QuestionText']) if x['problem_type']==7 else x['QuestionText'],axis=1)
agg_val['QuestionText']=agg_val['QuestionText'].apply(pf.strip_extra_whitespace)



In [18]:
# --- Assuming the setup code has been run and checkpoint_dir is defined ---
# project_dir = os.path.join(DRIVE_DIR, 'LLM_Misconception_Project')
# checkpoint_dir = os.path.join(project_dir, 'Checkpoints')
# --------------------------------------------------------------------------

print(f"Directory to check: {checkpoint_dir}")

# Check if the directory exists before trying to list its contents
if os.path.exists(checkpoint_dir):
    contents = os.listdir(checkpoint_dir)

    if contents:
        print(f"\n✅ Contents of '{os.path.basename(checkpoint_dir)}':")
        for item in contents:
            print(f"- {item}")
    else:
        print("\n⚠️ The checkpoint directory is empty.")
else:
    print("\n❌ Error: The checkpoint directory does not exist or the drive is not fully mounted.")

Directory to check: /content/drive/MyDrive/LLM_Misconception_Project/checkpoints

✅ Contents of 'checkpoints':
- validation_analysis
- validation_results
- stage2_explanation_model_dec5.pt
- combined_validation_results
- stage2_explanation_model_dec5_MISC_ONLY_SUB.pt
- stage2_explanation_model_dec5_MISC_ONLY_SUB_Other.pt
- stage2_explanation_model_dec5_MISC_ONLY_SUB_Other_B.pt
- stage2_explanation_model_dec5_restructure_B.pt
- stage2_explanation_model_dec5_restructure.pt
- other_samples_predicted.csv
- other_samples_promoted_to_training.csv
- other_samples_relabeled.csv


## 🔀 **Strategic Deduplication: Balancing Diversity and Coverage**

---

### **The Risk: Semantic Overfitting**

When generating thousands of synthetic samples from templates, there's a risk of creating **semantically identical explanations** that differ only in their numerical values. For example:

> "Since Chris eaten 3/5 of the pie, that is what remains and you then take 10/12 of that which means multiply the 2 fractions so 3/5 × 10/12 which equals 30/60"

Having this exact sentence structure repeated hundreds of times with different fractions (2/3 × 4/5, 1/4 × 7/8, etc.) could cause the model to **overfit to specific linguistic patterns** rather than learning generalizable reasoning structures.

---

### **The Solution: Structural Key Deduplication**

To address this, a **structural normalization** was applied:
```python
def create_structural_explanation_key(explanation_text):
    # Replace fractions (1/3, 3/8, etc.) with [FRAC]
    normalized_text = re.sub(r'\d+\s*/\s*\d+', '[FRAC]', explanation_text)
    
    # Replace remaining numbers with [NUM]
    normalized_text = re.sub(r'(\b\d+(\.\d*)?\b)|(\b\.\d+\b)', '[NUM]', normalized_text)
    
    # Normalize: lowercase, remove extra spaces
    normalized_text = normalized_text.lower()
    normalized_text = re.sub(r'\s+', ' ', normalized_text).strip()
    
    return normalized_text
```

This creates a **value-independent template** for each explanation, allowing deduplication based on sentence structure rather than specific numbers.

---

### **Controlled Sampling Strategy**

Rather than completely eliminating all duplicates, the dataset was **strategically sampled**:

1. **Initial deduplication:** Keep only one instance of each structural pattern
2. **Multiple random samples:** Draw from the data 3-6 times with different random states
3. **Special handling:** Ensure "Irrelevant" misconception samples get additional representation

This approach provides:
- **Diversity:** Avoid hundreds of semantically identical explanations
- **Coverage:** Retain a few variations (2-4 instances) of important patterns
- **Balance:** With feature substitution already in place, extensive repetition becomes unnecessary

**Result:** Original samples reduced from tens of thousands to a **deduplicated set** that maintains pattern diversity while avoiding semantic overfitting.

---

### **Why Limited Repetition Is Acceptable**

Since the training pipeline already includes:
- ✅ **Feature substitution** (numbers → placeholders like F1, N3)
- ✅ **TextAugmenter** (synonym replacement, misspellings, capitalization variations)

The model receives **implicit variation** even from structurally similar explanations, making aggressive deduplication both safe and beneficial for generalization.

In [18]:
def create_structural_explanation_key(explanation_text):
    """
    Normalizes the student explanation by replacing numbers with a [NUM] token
    to generate a unique structural key.

    Args:
        explanation_text (str): The raw student explanation.

    Returns:
        str: The canonical, value-independent explanation template.
    """
    # 1. Replace fractions (like 1/3, 3/8, 4/11) with a single token
    # Pattern: Digit(s), followed by /, followed by Digit(s)
    normalized_text = re.sub(r'\d+\s*/\s*\d+', '[FRAC]', explanation_text)

    # 2. Replace all remaining standalone numbers (integers and decimals)
    # This should catch 1, 2, 3, 5, 8, etc.
    # Pattern: Optional sign/decimal point, followed by one or more digits
    # Use word boundaries (\b) to avoid replacing numbers inside words (e.g., '1' in 'C1')
    normalized_text = re.sub(r'(\b\d+(\.\d*)?\b)|(\b\.\d+\b)', '[NUM]', normalized_text)

    # 3. Clean up the text: lowercase, remove extra spaces, and strip punctuation
    normalized_text = normalized_text.lower()
    normalized_text = re.sub(r'\s+', ' ', normalized_text).strip()

    return normalized_text


augmented_data_set_all=augment_functions.generate_all_augmented_data(change_cat_prob=0.00)
augmented_data_set_all['Structural_Key'] = augmented_data_set_all['StudentExplanation'].apply(create_structural_explanation_key)

do_deduplicate=True
if do_deduplicate:
  # 2. Deduplicate
  print(f"Original samples: {len(augmented_data_set_all)}")
  states=[i for i in range(100)]
  augmented_data_set=augmented_data_set_all.groupby(['Structural_Key']).first().reset_index()
  print('here')
  print(f"Deduplicated samples 1 time: {len(augmented_data_set)}")
  print()
  for state in range(41,43):
    augmented_data_set_all=augmented_data_set_all.sample(frac=1,random_state=states[state])
  # Group by the key and the Misconception (in case the same template applies to different problems)
    augmented_data_set = pd.concat([augmented_data_set, augmented_data_set_all.groupby(['Structural_Key']).first().reset_index()])
  for state in range(11,15):
    augmented_data_set_all=augmented_data_set_all.sample(frac=1,random_state=states[state])
    is_irrelevant = augmented_data_set_all['Misconception'].apply(
    lambda x: 'Irrelevant' in x
)

# 2. Use the boolean Series to filter the DataFrame
    irrelevant_set = augmented_data_set_all[is_irrelevant].copy()
    augmented_data_set = pd.concat([augmented_data_set, irrelevant_set.groupby(['Structural_Key']).first().reset_index()])

augmented_data_set['MC_Answer'] = augmented_data_set['MC_Answer'].apply(pf.standardize_answer_format_v5)
print(f"Deduplicated samples for Stage 2: {len(augmented_data_set)}")



Number of augment functions called 60
Generated 18 examples for augment_adding_across1
Generated 840 examples for augment_subtraction
Generated 18 examples for augment_inverse_operation
Generated 3049 examples for augment_longer_is_bigger
Generated 120 examples for augment_multiplying_by_4
Generated 760 examples for augment_fraction_word_problems
Generated 2730 examples for augment_swap_dividend
Generated 4284 examples for augment_mult_misconception
Generated 1418 examples for augment_flipchange_misconception
Generated 379 examples for augment_inversion_misconception
Generated 8464 examples for augment_duplication_misconception
Generated 951 examples for augment_wrong_operation_misconception
Generated 1560 examples for augment_firstterm_misconception
Generated 1125 examples for augment_base_rate
Generated 3600 examples for augment_division
Generated 1362 examples for augment_scale_misconception
Generated 1037 examples for augment_definition
Generated 1520 examples for augment_wnb
Gener

#

## 📊 **Assessment Integration: Building a Data Feedback Loop**

---

### **Purpose: Continuous Model Improvement**

The training pipeline includes infrastructure to **incorporate real assessment data** collected from students. This creates a feedback loop where:

1. Students complete assessments through a deployed application
2. Their responses are saved as structured assessment files
3. New data is automatically integrated into the training pipeline
4. Models are retrained with expanded real-world examples

Currently, the assessment files contain **self-typed responses** for testing and validation, but the system is designed to scale with data from actual student interactions.


### **Future Deployment Strategy**

The assessment integration system enables:

✅ **Scalable data collection:** Students complete assessments → responses automatically added to training corpus  
✅ **Real-world diversity:** Capture authentic student reasoning patterns beyond synthetic templates  
✅ **Continuous improvement:** Retrain models periodically with expanded datasets  
✅ **Quality control:** Label validation ensures new data maintains consistency with existing categories  

This infrastructure transforms the model from a **static classifier** into a **continuously learning system** that improves with each deployment cycle.

In [19]:
def fix_mislabeled_items(misc_list):
    # Dictionary of {Wrong: Right}
    corrections = {
        'Firsterm': 'Firstterm',
        'Flipchange': 'FlipChange',
        ' Incomplete': 'Incomplete',
        'Incomplete ': 'Incomplete',
        'Adding_Across': 'Adding_across'
    }

    new_list = []
    for item in misc_list:
        # Handle the specific case: 'Other, Incomplete' -> ['Other', 'Incomplete']
        if ',' in item:
            parts = [p.strip() for p in item.split(',')]
            for p in parts:
                corrected = corrections.get(p, p)
                new_list.append(corrected)
        else:
            # Standard cleanup for single items
            clean_item = item.strip()
            new_list.append(corrections.get(clean_item, clean_item))

    return new_list
all_files=os.listdir(data_dir)
assessment_files=[f for f in all_files if 'Assessment' in f or 'Assessement' in f]
all_assessment_dfs=pd.DataFrame()
for fname in assessment_files:
  assessment_df=pd.read_csv(os.path.join(data_dir,fname))
  assessment_df=assessment_df.dropna(subset=['QuestionText'])
  assessment_df['MC_Answer']=assessment_df['MC_Answer'].astype(str)
  try:
    assessment_df['Category'] = assessment_df['Category'].apply(ast.literal_eval)
    assessment_df['Misconception'] = assessment_df['Misconception'].apply(ast.literal_eval)
  except:
    print(fname)
  print(f"Loaded: {fname}")

  assessment_df['Misconception'] = assessment_df['Misconception'].apply(fix_mislabeled_items)
  assessment_df['fname']=fname

  all_assessment_dfs=pd.concat([all_assessment_dfs,assessment_df])
# Extract the first element from the Category list
all_assessment_dfs['Original_category'] = all_assessment_dfs['Category'].apply(lambda x: x[0])

# Standardize the math answer formatting
all_assessment_dfs['MC_Answer'] = all_assessment_dfs['MC_Answer'].apply(pf.standardize_answer_format_v5)

# Calculate binary correctness: 1 if the category starts with 'True', else 0
all_assessment_dfs['Correctness'] = all_assessment_dfs['Original_category'].apply(
    lambda x: 1 if x.split('_')[0].upper() == 'TRUE' else 0
)
all_assessment_dfs['function']='self_typed'
all_assessment_dfs['Structural_Key']='None'
all_assessment_dfs=all_assessment_dfs[['Structural_Key', 'QuestionText', 'StudentExplanation', 'Category',
       'Misconception', 'MC_Answer', 'function', 'problem_type',
       'Correctness','fname']]
count=0
for ind, row in all_assessment_dfs.iterrows():
  misc=row['Misconception']
  category=row['Category']
  for item in misc:
    if item not in label_encoder_misconception.classes_:
      print(f"Misconception: {item}")
      print(f"File: {row['fname']}")
      count+=1
  for item in category:
    if category not in label_encoder_category.classes_:
      count+=1
      print(f"Category: {item}")
      print(f"File: {row['fname']}")
if count==0:
  print('All assessments contain proper labels')
all_assessment_dfs.head()



Loaded: Assessment_1.csv
Loaded: Assessment_3.csv
Loaded: Assessment_4.csv
Loaded: Assessment_2.csv
Loaded: Assessment_5.csv
Loaded: Assessment_6.csv
Loaded: Assessment_7.csv
Loaded: Assessment_8.csv
Loaded: Assessment_9.csv
Loaded: Assessment_10.csv
Loaded: Assessment_11.csv
Loaded: Assessment_12.csv
Loaded: Assessment_14.csv
Loaded: Assessment_13.csv
Loaded: Assessment_15.csv
Loaded: Assessment_16.csv
Loaded: Assessment_17.csv
Loaded: Assessment_18.csv
Loaded: Assessment_19.csv
Loaded: Assessment_20.csv
Loaded: Assessment_other.csv
Loaded: Assessment_other2.csv
Loaded: Assessment_23.csv
Loaded: Assessment_22.csv
Loaded: Assessment_21.csv
Loaded: Assessment_24.csv
Loaded: Assessment_25.csv
Loaded: Assessment_26.csv
Loaded: Assessment_28.csv
Loaded: Assessment_39.csv
Loaded: Assessment_40.csv
Loaded: Assessment_29.csv
Loaded: Assessment_30.csv
Loaded: Assessment_31.csv
Loaded: Assessment_32.csv
Loaded: Assessment_33.csv
Loaded: Assessment_34.csv
Loaded: Assessment_35.csv
Loaded: Assess

,Structural_Key,QuestionText,StudentExplanation,Category,Misconception,MC_Answer,function,problem_type,Correctness,fname
0,None,Compute \(\frac{1}{3} \div 6\). reduce if nece...,I treated it as FlipChange_UNSIM2 so FlipChang...,[False_Misconception],[FlipChange],\( 2 \),self_typed,1.0,0,Assessment_1.csv
1,None,Dots have been arranged in these patterns: [Im...,well its increasing by SEQ_COMMON_DIFF so the ...,[True_Correct],[No Misconception],\( \frac{5}{4} \),self_typed,2.0,1,Assessment_1.csv
2,None,Calculate \(\frac{2}{3} \times 15\). reduce if...,times N3 by F1_NUM and N3 by F1_DEN,[False_Misconception],"[Duplication, Incomplete]",\( \frac{30}{45} \),self_typed,3.0,0,Assessment_1.csv
3,None,A bag contains \( 21 \) red and white marbles....,take FRACTION of ALL_UNITS. ALL_UNITS divide F...,[False_Misconception],[Wrong_fraction],\( 6 \),self_typed,4.0,0,Assessment_1.csv
4,None,\( \frac{1}{3}+\frac{2}{5}= \),F1_ORIG_NUM + F2_ORIG_NUM IS F1_ORIG_DEN AND F...,[False_Misconception],[Adding_across],\( \frac{3}{8} \),self_typed,5.0,0,Assessment_1.csv


In [20]:
augmented_data_set['Original_category'] = augmented_data_set['Category'].apply(lambda x: x[0])
augmented_data_set['MC_Answer'] = augmented_data_set['MC_Answer'].apply(pf.standardize_answer_format_v5)
augmented_data_set['Correctness']=augmented_data_set['Original_category'].apply(lambda x: 1 if x.split('_')[0] in ['True','TRUE'] else 0)

df_filtered = augmented_data_set[
    (augmented_data_set['Misconception'].apply(lambda x: 'No Misconception' not in x)) &
    (augmented_data_set['Misconception'].apply(lambda x: 'Other' not in x))&
    (augmented_data_set['Misconception'].apply(lambda x: 'Incomplete' not in x))
]

neither_samples = create_neither_samples(
    df=df_filtered,  # or augmented_data_set
    sample_fraction=0.3,     # Convert 30% of misconception samples
    seed=42
)
neither_samples['Correctness']=neither_samples['Original_category'].apply(lambda x: 1 if x.split('_')[0] == 'True' else 0)
data_for_cat_aug=augmented_data_set[(augmented_data_set['problem_type']!=10)&(augmented_data_set['Misconception']!='Unknowable')]
changed_cats=CategoryAugmenter(data_for_cat_aug).augment_correct_data()

changed_cats=changed_cats.sample(frac=0.4)
augment=True
augmented_only=False
other_samples=False
use_neither_samples=True
do_changed_cats=False
use_test_in_train=True


#other_df_agg_train=other_df_agg_train.sample(frac=0.5)
#other_df_agg_val=other_df_agg_val.sample(frac=0.5)
neither_samples=neither_samples.sample(frac=0.5)
neither_samples=neither_samples.sample(frac=0.4)
if augment:
    agg_train_reduced=agg_train.drop(columns=['tuple_key','row_id'])
    final_training_set=pd.concat([augmented_data_set[agg_train_reduced.columns],agg_train_reduced])
    if do_changed_cats:
        final_training_set=pd.concat([final_training_set,changed_cats])

    agg_val=pd.concat([agg_val_copy,augmented_data_set_val[agg_train_reduced.columns]])

else:
    if augmented_only:
        final_training_set=augmented_data_set[agg_train.columns]
    else:
      final_training_set=agg_train
if other_samples:
    final_training_set=pd.concat([final_training_set,other_df_agg_train.sample(frac=0.5)])
    agg_val=pd.concat([agg_val,other_df_agg_val.sample(frac=0.5)])
    if use_test_in_train:
      final_training_set=pd.concat([final_training_set,other_df_agg_test])
if use_neither_samples:
  final_training_set=pd.concat([final_training_set,neither_samples.sample(frac=0.5)])

if use_test_in_train:
  final_training_set=pd.concat([final_training_set,agg_test])
final_training_set=pd.concat([final_training_set,all_assessment_dfs])
final_training_set


📊 Creating 'Neither' samples:
  Total misconception samples: 18170
  Converting 5451 samples (30.0%)

📝 Explanation type distribution:
  Empty/blank: 3112 (18.8%)
  Random guess: 5259 (31.7%)
  Letter preference: 1864 (11.2%)
  Vague: 6359 (38.3%)

📊 Category distribution:
  False_Neither: 15990
  True_Neither: 604


,QuestionText,StudentExplanation,MC_Answer,Category,Misconception,Correctness,problem_type,Structural_Key,function,Original_category,row_id,tuple_key
0,\( \frac{-11}{12} - \frac{-15}{4} = \),(-11/12)+(15/4)=17/6,\( \frac{17}{6} \),[True_Correct],[No Misconception],1,12,NaN,NaN,NaN,NaN,NaN
1,What is (-9)-(-14)?,(-9)+(14)=5,\( 5 \),[True_Correct],[No Misconception],1,12,NaN,NaN,NaN,NaN,NaN
2,\( \frac{3}{2} + \frac{4}{8} = \),(3+4)/8 so 7/8,\( \frac{7}{8} \),[False_Misconception],[Denominator-only_change],0,5,NaN,NaN,NaN,NaN,NaN
3,\( \frac{3}{4} + \frac{3}{8} = \),(3+3)/8 so 6/8 This simplifies to 3/4,\( \frac{3}{4} \),[False_Misconception],[Denominator-only_change],0,5,NaN,NaN,NaN,NaN,NaN
4,\( \frac{4}{2} + \frac{6}{10} = \),(4+6)/10 so 10/10 This simplifies to 1,\( \frac{1}{1} \),[False_Misconception],[Denominator-only_change],0,5,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
24282,\( \frac{A}{10}=\frac{9}{15} \) What is the va...,you do what do you do to the bottom to the top...,\( 6 \),[True_Misconception],[Additive],1,14,NaN,NaN,NaN,[14256],(\( \frac{A}{10}=\frac{9}{15} \) What is the v...
24291,\( \frac{A}{10}=\frac{9}{15} \) What is the va...,you need to divide the second fraction by 3 an...,\( 6 \),[True_Correct],[No Misconception],1,14,NaN,NaN,NaN,[14264],(\( \frac{A}{10}=\frac{9}{15} \) What is the v...
24295,\( \frac{A}{10}=\frac{9}{15} \) What is the va...,you put them both to 30 (denominator) and get ...,\( 6 \),[True_Correct],[No Misconception],1,14,NaN,NaN,NaN,[14267],(\( \frac{A}{10}=\frac{9}{15} \) What is the v...
24297,\( \frac{A}{10}=\frac{9}{15} \) What is the va...,you times the denominator to make them the sam...,\( 6 \),[True_Correct],[No Misconception],1,14,NaN,NaN,NaN,[14268],(\( \frac{A}{10}=\frac{9}{15} \) What is the v...


In [21]:
important=final_training_set[final_training_set['problem_type']==3]
final_training_set=final_training_set[final_training_set['problem_type']!=3]
final_training_set=final_training_set.groupby('problem_type').sample(500)
final_training_set=pd.concat([final_training_set,important])
final_training_set=final_training_set.reset_index(drop=True)
final_training_set=pd.concat([final_training_set,all_assessment_dfs]).reset_index(drop=True)


In [22]:
# More Cleaning
final_training_set['Category']=final_training_set['Category'].apply(lambda x: x[0])
final_training_set['Category']=final_training_set['Category'].str.replace('TRUE','True')
final_training_set['Category'].value_counts()

,count
Category,
False_Misconception,9312
True_Correct,5980
False_Neither,489
True_Misconception,176
False_Correct,32
True_Neither,13


## ⚙️ **Feature Engineering: Creating Mathematical Context**

---

This extensive code block implements the **`create_generic_math_context`** function, which is the core of your feature engineering process. This function analyzes the problem text and the multiple-choice answer to generate a comprehensive set of mathematical relationships and potential misconception triggers.

The strategy here is **gated feature generation**. Instead of creating every possible feature for every question, the `categorize_question` function first determines the **problem type** (e.g., fraction, decimal, angle, integer) and then the feature generation is restricted to relevant mathematical domains.

### **Specific Features Developed for Misconceptions**

The code generates numerous features designed to flag numerical results commonly produced by specific student misconceptions. These features are concatenated into a long string (the `Mathematical_Context`) and will be passed to your model as text, allowing the LLM to learn the association between the student's *explanation* and these *mathematical results*.

| Problem Type | Feature Examples | Misconception Targeted |
| :--- | :--- | :--- |
| **Decimals/Comparison** | `N1_MORE_DECIMAL_PLACES_THAN_N2:TRUE` | Errors related to treating decimals like integers (e.g., believing 0.5 is smaller than 0.09 because $5 < 9$). |
| | `NUM_LARGEST_NON_ZERO_DECDIGIT` | Compares the decimal part as if it were an integer (e.g., comparing 79 in 6.079 vs 3 in 6.3). |
| | `N1_LEADING_ZEROS:X`, `N1_IS_GREATER_THAN_0_5:TRUE` | Helps flag values commonly used in rounding or probability tasks. |
| **Fractions/Division** | `N{j}divF{i}_SWAPPED_DEC:X` | **Swap Dividend/Divisor:** Computes the result of dividing the numbers in the wrong order (e.g., $N \div F$ instead of $F \div N$). |
| | `F{i}_BOTH_MULT_N{j}:num/den` | **Duplication/Common Denominator Error:** Computes the result of multiplying both the numerator and denominator by a whole number, which is a common mistake when finding a common denominator. |
| | `FRAC_DEN_LCM:X`, `FRAC_DEN_GCD:Y` | Provides critical information for common denominator and simplification errors. |
| **Equations/Algebra** | `EQ_LAST_DIGIT_MISC:X`, `EQ_TAIL_DIGITS_MISC:Y` | **Ignoring Place Value:** Flags results from ignoring digits (e.g., solving $20 + x = 26$ and answering '6' because they ignore the '2' in $20$). |
| **Integers** | `N1_N2_ABS_SUM:X`, `N1_N2_ABS_DIFF:Y` | **KCC (Keep, Change, Change):** Provides the sums and differences of absolute values, which are key intermediate results in integer subtraction misconceptions. |
| **Angles/Geometry** | `N{i}_SIDES_FROM_EXT:X`, `N{i}_EXTERIOR_ANGLE:Y` | **Formula Confusion:** Provides results from common geometry formulas (e.g., calculating sides from the interior angle, $360 / (180 - I)$), helping identify formula misuse. |

### **The Final Step**

The last lines apply this function to all splits (`final_training_set`, `agg_val`, etc.), appending the resulting feature string (e.g., `| SA:1/2 | NUM_COUNT:2 | F1_RECIP:2/1 | ... |`) to the new **`Mathematical_Context`** column. This column will be combined with the `QuestionText` and `StudentExplanation` to form the full input for your model.

In [23]:
final_training_set['Mathematical_Context']=final_training_set[['QuestionText','MC_Answer','problem_type']].apply(lambda x: augment_functions.create_generic_math_context(x['QuestionText'],x['MC_Answer'],x['problem_type']),axis=1)
agg_val['Mathematical_Context']=agg_val[['QuestionText','MC_Answer','problem_type']].apply(lambda x: augment_functions.create_generic_math_context(x['QuestionText'],x['MC_Answer'],x['problem_type']),axis=1)

final_training_set['QuestionText']=final_training_set.apply(lambda x: pf.rephrase_question_and_explanation(x['QuestionText']) if x['problem_type']==7 else x['QuestionText'],axis=1)
final_training_set['StudentExplanation'] = final_training_set['StudentExplanation'].apply(augmenter.clean)
agg_val['StudentExplanation'] = agg_val['StudentExplanation'].apply(augmenter.clean)
final_training_set['QuestionText']=final_training_set['QuestionText'].apply(pf.strip_extra_whitespace)
agg_val['QuestionText']=agg_val['QuestionText'].apply(pf.strip_extra_whitespace)

other_df_agg['QuestionText']=other_df_agg['QuestionText'].apply(pf.strip_extra_whitespace)
other_df_agg['StudentExplanation'] = other_df_agg['StudentExplanation'].apply(augmenter.clean)
other_df_agg['Mathematical_Context']=other_df_agg[['QuestionText','MC_Answer','problem_type']].apply(lambda x: augment_functions.create_generic_math_context(x['QuestionText'],x['MC_Answer'],x['problem_type']),axis=1)



In [24]:
# Fill in missing problem_type values
final_training_set['problem_type']=final_training_set['problem_type'].fillna(1)
final_training_set['problem_type']=final_training_set['problem_type'].astype(int)
agg_val['problem_type']=agg_val['problem_type'].fillna(1)
agg_val['problem_type']=agg_val['problem_type'].astype(int)
final_training_set['Correctness'].isnull().sum()
final_training_set['problem_type'].isnull().sum()

np.int64(0)

## 🔄 **Mathematical Context Substitution: The Key to Robust Generalization**

---

### **The Problem: Value Inconsistency Obscures Language Patterns**

During initial model development, a critical challenge emerged: the model struggled to make accurate predictions on new data, particularly when students typed their own explanations with different numerical values than those seen during training. The model was **overfitting to specific numerical values** rather than learning the underlying linguistic and mathematical reasoning patterns.

For example, when trying to detect the "Multiplication" misconception (treating division as multiplication), the model would see training examples like:
- "1/2 of 6 is 6/2 which equals 3"
- "1/3 of 9 is 9/3 which equals 3"  
- "1/4 of 28 is 28/4 which equals 7"

While all these examples share the same **linguistic pattern** (the word **"of"** indicating multiplication), the varying numerical values distracted the model from recognizing this consistent language cue.

---

### **The Solution: Feature-Based Substitution**

To address this, every numerical value and computed result in the student explanation was **systematically replaced with named feature placeholders** derived from the mathematical context extraction process.

**Example Transformation:**

**Question:**  
> Calculate \(\frac{1}{7} \div 14\). Reduce if necessary.

**Original Student Explanation:**  
> "I thought the answer is 2 because 1/7 of 14 is 14/7 which equals 2."

**Feature-Substituted Explanation:**  
> "I thought the answer is F1xN3_SwapDividend_MIXED because F1 of N3 is F1xN3_SwapDividend_UNSIM which equals F1xN3_SwapDividend_MIXED."

**Predicted Misconception:** `Mult` (Multiplication) ✅

---

### **Why This Works: Focus on Language, Not Numbers**

After substitution, the model sees **consistent patterns** across all examples:

- `F1 of N3` instead of varying combinations like "1/2 of 6", "1/3 of 9", "1/4 of 28"
- The critical linguistic marker **"of"** remains unchanged and now consistently appears between the same placeholder types
- The model learns that `F1 of N3` → Multiplication misconception, regardless of the actual numerical values

This approach creates a **controlled vocabulary for mathematical values**, allowing the model to:

1. **Recognize linguistic patterns:** Words like "of", "divided by", "times", "minus" become the primary signals
2. **Ignore numerical noise:** The specific values (2, 14, 1/7) don't distract from the reasoning structure
3. **Generalize robustly:** Predictions remain accurate even on completely new numerical combinations or user-typed explanations

---

### **Impact on Model Performance**

Before implementing mathematical context substitution, the models produced **inconsistent and unreliable predictions** on new data, especially self-typed student responses.

After substitution, the models achieved:
- **Robust inference** on unseen numerical values
- **Accurate predictions** on user-generated explanations
- **Consistent performance** across the entire validation set
- **High F1 scores:** 0.9828 (Correctness) and 0.9716 (Misconception)

The substitution technique proved to be **essential** for building models that understand mathematical reasoning patterns rather than memorizing specific numerical examples.

In [25]:
def apply_feature_substitution(row):
  explanation=row['StudentExplanation']
  math_context_string=row['Mathematical_Context']
  problem_type=row['problem_type']
  feature_map, value_map = augment_functions.parse_and_map_features(math_context_string)
  new_explanation = augment_functions.place_generic_features_in_text(value_map, explanation,problem_type)
  return new_explanation
final_training_set['Original_Explanation']=final_training_set['StudentExplanation']
final_training_set['StudentExplanation']=final_training_set.apply(apply_feature_substitution,axis=1)
agg_val['StudentExplanation']=agg_val.apply(apply_feature_substitution,axis=1)
other_df_agg['Original_Explanation']=other_df_agg['StudentExplanation']
other_df_agg['StudentExplanation']=other_df_agg.apply(apply_feature_substitution,axis=1)

### **View Explanation Before and After Feature Substitution**

In [43]:
def inspect_training_samples(df, n=5):
    """
    Prints specific columns from a random sample of the dataframe
    to verify text content and labels.
    """
    print(f"\n{'='*30} INSPECTING {n} SAMPLES {'='*30}\n")

    # Randomly sample the dataframe
    samples = df.sample(min(n, len(df)))

    for i, (idx, row) in enumerate(samples.iterrows(), 1):
        print(f"SAMPLE #{i} (Index: {idx})")
        print(f"  [QUESTION]:      {row.get('QuestionText', 'N/A')}")
        print(f"  [MC ANSWER]:     {row.get('MC_Answer', 'N/A')}")

        # Format Misconception in case it is still a list
        misc = row.get('Misconception', 'N/A')
        misc_str = ", ".join(misc) if isinstance(misc, list) else str(misc)
        print(f"  [MISCONCEPTION]: {misc_str}")

        print(f"  [STUDENT EXPL]:  {row.get('StudentExplanation', 'N/A')}")
        print(f"  [ORIGINAL EXPL]: {row.get('Original_Explanation', 'N/A')}")
        print("-" * 80)

# Execute the function
inspect_training_samples(final_training_set, n=2)


============================== INSPECTING 2 SAMPLES ==============================

SAMPLE #1 (Index: 605)
  [QUESTION]:      Robert has \( \frac{1}{3} \) of a whole cake in the fridge. Melissa eats \( \frac{1}{5} \) of this piece. What fraction of the whole cake has Melissa eaten? Choose the number sentence that would solve the word problem.
  [MC ANSWER]:     \( \frac{1}{3} \div \frac{1}{5} \)
  [MISCONCEPTION]: Division
  [STUDENT EXPL]:  F2 divided by F1 would give you the answer as Melissa has F1 of what Robert has. Robert has F2.
  [ORIGINAL EXPL]: 1/3 divided by 1/5 would give you the answer as Melissa has 1/5 of what Robert has. Robert has 1/3.
--------------------------------------------------------------------------------
SAMPLE #2 (Index: 3692)
  [QUESTION]:      Maria has 3/12 of a roll of ribbon. John uses 1/6 of the ribbon. What fraction of the whole roll did John use?
  [MC ANSWER]:     \( \frac{3}{2} \)
  [MISCONCEPTION]: Division
  [STUDENT EXPL]:  do N2/N2divF2_SWAPP

## 🎯 **Feature Reduction: Filtering Signal from Noise**

---

### **The Challenge: Too Much Information**

After generating comprehensive mathematical context features for each question, the feature strings became **extremely verbose**, containing dozens of numerical values, ratios, and computed results. While this information was mathematically complete, it introduced significant **noise** that obscured the truly relevant patterns.

**Example - Full Context (Probability Question):**
```
SA:Likely | NUM_COUNT:1 | FRAC_COUNT:0 | N1:0.9 | N1_DECIMAL_PLACES:1 |
N1_LEADING_ZEROS:0 | N1_SIG_DECIMAL:9 | N1_FIRST_NON_ZERO_DEC_DIGIT:9 |
N1_DECIMAL_PART:9 | N1_IS_GREATER_THAN_0_5:True | N1_IS_LESS_THAN_0_5:False |
MAX_DECIMAL_PLACES_COUNT:1 | LONGEST_DECIMAL_VALUE:0.9 |
LONGEST_DECIMAL_PLACE_NAME:tenth | MIN_DECIMAL_PLACES_COUNT:1 |
SHORTEST_DECIMAL_VALUE:0.9 | SHORTEST_DECIMAL_PLACE_NAME:tenth |
ACTUAL_LARGEST_VALUE:0.9 | ACTUAL_LARGEST_PLACE_NAME:tenth |
ACTUAL_SMALLEST_VALUE:0.9 | ACTUAL_SMALLEST_PLACE_NAME:tenth |
N1_IS_LONGEST_DECIMAL:TRUE | N1_IS_SHORTEST_DECIMAL:TRUE |
N1_IS_ACTUAL_LARGEST:TRUE | N1_IS_ACTUAL_SMALLEST:TRUE |
WHOLE_NUMBER_PRESENT:False | ANSWER_HAS_FRACTION:FALSE |
PROBABILITY_GREATER_THAN_0.5: 0.9 | COMP_LESS_THAN_0.5: 0.1 |
COMP_PERCENT_LESS_THAN_50: 10 | PERCENT_GREATER_THAN_50: 90
```

This level of detail was **counterproductive** - the model had to parse through numerous irrelevant numerical values to find the meaningful relationships.

---

### **The Solution: Boolean Features Only**

To address this, the mathematical context was **filtered to include only boolean (TRUE/FALSE) features**. These features capture **relational properties and comparisons** that are likely to appear in student explanations, without the distraction of specific numerical values.
```python
def filter_math_context_case_insensitive(context_string):
    if not isinstance(context_string, str):
        return ""
    
    # Extract only features with TRUE/FALSE values
    pattern = r'\w+\s*:\s*(?:TRUE|FALSE)'
    boolean_features = re.findall(pattern, context_string, flags=re.I)
    
    return " | ".join(boolean_features)
```

**Reduced Context (Same Question):**
```
N1_IS_GREATER_THAN_0_5:True | N1_IS_LESS_THAN_0_5:False |
N1_IS_LONGEST_DECIMAL:TRUE | N1_IS_SHORTEST_DECIMAL:TRUE |
N1_IS_ACTUAL_LARGEST:TRUE | N1_IS_ACTUAL_SMALLEST:TRUE |
WHOLE_NUMBER_PRESENT:False | ANSWER_HAS_FRACTION:FALSE
```

---

### **Why Boolean Features Are Optimal**

Boolean features capture **exactly the type of information** students reference in their explanations:

**Example - Decimal Comparison Question:**

**Question:** Select the lesser number: 32 and 32.08

**Reduced Context:**
```
ANSWER_MATCHES_HIGHEST_NON_ZERO_DIGIT:False |
ANSWER_MATCHES_LOWEST_NON_ZERO_DIGIT:True |
PICKED_HIGHEST_DIGIT_NOT_VALUE:FALSE |
N1_LESS_DECIMAL_PLACES_THAN_N2:TRUE |
N1_LT_N2:TRUE |
N1_IS_SHORTEST_DECIMAL:TRUE |
N1_IS_ACTUAL_SMALLEST:TRUE |
N2_IS_LONGEST_DECIMAL:TRUE |
N2_IS_ACTUAL_LARGEST:TRUE |
WHOLE_NUMBER_PRESENT:TRUE
```

These features encode the **relationships** students might reference:
- "32 has fewer decimal places than 32.08" → `N1_LESS_DECIMAL_PLACES_THAN_N2:TRUE`
- "32 is the shortest decimal" → `N1_IS_SHORTEST_DECIMAL:TRUE`
- "32.08 is actually larger" → `N2_IS_ACTUAL_LARGEST:TRUE`

Without the noise of specific values like "N1_DECIMAL_PART:8" or "N1_DIV_N2_RATIO:0.997506", the model can focus on these **meaningful relational properties**.

---

### **Impact on Model Performance**

Reducing features to boolean-only values resulted in:

✅ **Reduced noise:** Eliminated hundreds of irrelevant numerical values per sample  
✅ **Clearer signal:** Model focuses on relational properties that actually appear in explanations  
✅ **Improved generalization:** Less overfitting to specific numerical patterns  
✅ **Faster training:** Shorter input sequences reduce computational overhead  
✅ **Better predictions:** Model performance improved across both correctness and misconception classification

The combination of:
1. **Feature substitution** in explanations (F1, N1, etc.)
2. **Boolean-only context** features

Created the optimal balance between providing **necessary mathematical information** while avoiding **numerical noise** that would distract the model from learning true linguistic and reasoning patterns.

In [26]:
def filter_math_context_case_insensitive(context_string):
    if not isinstance(context_string, str):
        return ""

    # Regex breakdown:
    # \w+          -> The feature name (letters/numbers/underscores)
    # \s*:\s* -> The colon, allowing for optional spaces around it
    # (?:TRUE|FALSE) -> The literal words TRUE or FALSE
    # flags=re.I   -> Makes the whole search ignore case (T/t, R/r, etc.)

    pattern = r'\w+\s*:\s*(?:TRUE|FALSE)'
    boolean_features = re.findall(pattern, context_string, flags=re.I)

    # Re-join with pipes for a clean, scannable string
    return " | ".join(boolean_features)


final_training_set['Mathematical_Context_Reduced'] = final_training_set['Mathematical_Context'].apply(filter_math_context_case_insensitive)
agg_val['Mathematical_Context_Reduced'] = agg_val['Mathematical_Context'].apply(filter_math_context_case_insensitive)

In [45]:
def inspect_filtered_samples(df, name="Dataset", n=5):
    print(f"\n{'='*20} {name}: {n} Random Samples {'='*20}")

    # Selecting n random rows to compare the columns
    samples = df.sample(n)

    for i, (idx, row) in enumerate(samples.iterrows(), 1):
        full = str(row['Mathematical_Context']).strip()
        reduced = str(row['Mathematical_Context_Reduced']).strip()

        print(f"Sample {i} (Index: {idx})")
        print(f"Question: {row['QuestionText']}")
        print(f"  [FULL]:    {full if full else '[EMPTY]'}")
        print(f"  [REDUCED]: {reduced if reduced else '[EMPTY]'}")
        print("-" * 30)

    print(f"{'='*60}\n")

# Run inspection for both sets
inspect_filtered_samples(final_training_set[final_training_set['problem_type']==7], "FINAL TRAINING SET")
inspect_filtered_samples(agg_val, "AGGREGATED VALIDATION SET")


==================== FINAL TRAINING SET: 5 Random Samples ====================
Sample 1 (Index: 5999)
Question: Which number is the biggest? 4.03, 4.0002, and 4.000009
  [FULL]:    | SA:4.000009 | NUM_COUNT:3 | FRAC_COUNT:0 | N1:4.000009 | N2:4.0002 | N3:4.03 | N1_DECIMAL_PLACES:6 | N1_LEADING_ZEROS:5 | N1_SIG_DECIMAL:000009 | N1_FIRST_NON_ZERO_DEC_DIGIT:9 | N2_DECIMAL_PLACES:4 | N2_LEADING_ZEROS:3 | N2_SIG_DECIMAL:0002 | N2_FIRST_NON_ZERO_DEC_DIGIT:2 | N3_DECIMAL_PLACES:2 | N3_LEADING_ZEROS:1 | N3_SIG_DECIMAL:03 | N3_FIRST_NON_ZERO_DEC_DIGIT:3 | HIGHEST_VALUE_FIRST_NON_ZERO_DIGIT:3 | LOWEST_VALUE_FIRST_NON_ZERO_DIGIT:9 | MIDDLE_VALUE_FIRST_NON_ZERO_DIGIT:2 | ANSWER_FIRST_NON_ZERO_DIGIT:9 | ANSWER_MATCHES_HIGHEST_NON_ZERO_DIGIT:False | ANSWER_MATCHES_LOWEST_NON_ZERO_DIGIT:True | PICKED_HIGHEST_DIGIT_NOT_VALUE:TRUE | PICKED_HIGHEST_DIGIT_AND_VALUE:FALSE | ANSWER_DIGIT_GT_CORRECT_DIGIT:TRUE | N1_MORE_DECIMAL_PLACES_THAN_N2:TRUE | N1_MORE_LEADING_ZEROS_THAN_N2:TRUE | N1_MORE_DECIMAL_PLAC

# **Training Value Counts**

In [27]:
def true_vcs(vcs,classes):
    new_vcs={cls:0 for cls in classes}
    for classes, count in vcs.items():
        for cls in classes:
            new_vcs[cls]+=count
    return new_vcs
def min_value_count(true_vcs):
    return min(true_vcs.values())

vcs=final_training_set['Misconception'].value_counts()
true_value_counts=true_vcs(vcs,label_encoder_misconception.classes_)
for key, value in true_value_counts.items():
    print(key, value)

Adding_across 106
Adding_terms 49
Additive 177
Base_rate 156
Certainty 95
Definition 70
Denominator-only_change 197
Division 61
Duplication 3577
Firstterm 82
FlipChange 47
Ignores_zeroes 68
Incomplete 1789
Incorrect_equivalent_fraction_addition 16
Interior 157
Inverse_operation 65
Inversion 1800
Irrelevant 279
Longer_is_bigger 61
Mult 225
Multiplying_by_4 89
No Misconception 6026
Not_variable 107
Other 505
Positive 191
Scale 144
Shorter_is_bigger 34
Subtraction 152
SwapDividend 57
Tacking 116
Unknowable 67
WNB 130
Whole_numbers_larger 98
Wrong_Fraction 75
Wrong_Operation 122
Wrong_fraction 79
Wrong_term 103


In [29]:
final_training_set['Misconception'].value_counts()

,count
Misconception,
[No Misconception],5977
[Duplication],2647
[Inversion],1688
"[Incomplete, Duplication]",892
[Incomplete],713
...,...
"[SwapDividend, Incomplete]",1
"[WNB, Incomplete]",1
"[Duplication, Firstterm]",1


# **Check Corpus Length**

In [28]:
# --- Function to Count Words as Tokens ---

def count_tokens_proxy(text):
    """Counts words/space-separated units as a simple proxy for token count."""
    if not text:
        return 0
    # Split by any whitespace and filter out empty strings
    return len([word for word in str(text).split() if word])

# --- Execution ---

# ⚠️ NOTE: Ensure your 'final_training_set' DataFrame is loaded before running this code.

# 1. Create a copy and randomize the order
# Using frac=1 randomizes the entire DataFrame
randomized_df = final_training_set.sample(frac=1, random_state=42).reset_index(drop=True)

# 2. Prepare to store counts
token_counts = []
num_samples = min(100, len(randomized_df))

print(f"Sampling and concatenating the first {num_samples} rows...")
print("-" * 50)

# 3. Loop through samples and concatenate/count
for ind in range(num_samples):
    row = randomized_df.iloc[ind]

    # Concatenate the required fields into the model's single input sequence
    full_sequence = (
        row['Mathematical_Context'] + " " +
        row['QuestionText'] + " " +
        row['MC_Answer'] + " " +
        row['StudentExplanation']
    )

    # Count the words/proxy tokens
    count = count_tokens_proxy(full_sequence)
    token_counts.append(count)

    # Print the result
    print(f"Sample {ind+1}: {count} tokens (proxy)")
    if count>450:
      print("Full Sequence:")
      print(full_sequence)
      print(row['StudentExplanation'])
      print(row['problem_type'])

print("-" * 50)
print(f"Summary (based on {num_samples} samples):")
print(f"Max Proxy Token Count: {max(token_counts)}")
print(f"Average Proxy Token Count: {sum(token_counts) / len(token_counts):.2f}")

Sampling and concatenating the first 100 rows...
--------------------------------------------------
Sample 1: 157 tokens (proxy)
Sample 2: 180 tokens (proxy)
Sample 3: 177 tokens (proxy)
Sample 4: 145 tokens (proxy)
Sample 5: 174 tokens (proxy)
Sample 6: 112 tokens (proxy)
Sample 7: 105 tokens (proxy)
Sample 8: 168 tokens (proxy)
Sample 9: 239 tokens (proxy)
Sample 10: 70 tokens (proxy)
Sample 11: 162 tokens (proxy)
Sample 12: 101 tokens (proxy)
Sample 13: 124 tokens (proxy)
Sample 14: 90 tokens (proxy)
Sample 15: 144 tokens (proxy)
Sample 16: 103 tokens (proxy)
Sample 17: 101 tokens (proxy)
Sample 18: 77 tokens (proxy)
Sample 19: 78 tokens (proxy)
Sample 20: 167 tokens (proxy)
Sample 21: 400 tokens (proxy)
Sample 22: 187 tokens (proxy)
Sample 23: 404 tokens (proxy)
Sample 24: 176 tokens (proxy)
Sample 25: 169 tokens (proxy)
Sample 26: 171 tokens (proxy)
Sample 27: 163 tokens (proxy)
Sample 28: 175 tokens (proxy)
Sample 29: 143 tokens (proxy)
Sample 30: 126 tokens (proxy)
Sample 31: 14

# **Balance Dataset by Undersampling**
- This was not used for training, But is kept as an option.

In [48]:
def balance_training_set_by_type(df, mapping, anchor_classes=None):
    """
    Balances each problem type by capping majority classes at the count of the
    smallest 'main' misconception, while keeping minority classes as-is.
    """
    if anchor_classes is None:
        # These are the classes we usually use to 'set the floor'
        anchor_classes = ['No Misconception', 'Other', 'Irrelevant', 'Incomplete']

    # Helper to count lists/sets as unique keys
    df['temp_key'] = df['Misconception'].apply(lambda x: "|".join(sorted(x)))

    balanced_segments = []

    for pt in df['problem_type'].unique():
        pt_df = df[df['problem_type'] == pt]
        counts = pt_df['temp_key'].value_counts()

        # 1. Identify "Main" misconceptions for this problem type from your mapping
        # We ignore background classes (No Misconception, etc.) to find the 'true' bottleneck
        type_main_misconceptions = [
            m for m in mapping[pt]
            if m not in anchor_classes and f"{m}" in pt_df['temp_key'].values
        ]

        # 2. Find the Threshold (The "320" logic)
        # We look for the minimum count among the standalone main misconceptions
        if type_main_misconceptions:
            # Get counts for single-label versions of these misconceptions
            main_counts = [counts.get(m, 999999) for m in type_main_misconceptions]
            threshold = min(main_counts)
        else:
            # Fallback if it's a weird type: use the median or a fixed floor
            threshold = counts.median()

        print(f"Problem Type: {pt} | Calculated Cap (Threshold): {int(threshold)}")

        # 3. Apply the Cap: Sample down if count > threshold, else keep all
        for m_key, count in counts.items():
            class_subset = pt_df[pt_df['temp_key'] == m_key]

            if count > threshold:
                # Sample down the majority (e.g., Mult 1973 -> 320)
                balanced_segments.append(class_subset.sample(n=int(threshold), random_state=42))
            else:
                # Keep the minority (e.g., Incomplete 112 -> 112)
                balanced_segments.append(class_subset)

    # Reconstruct the dataframe
    new_df = pd.concat(balanced_segments).drop(columns=['temp_key']).reset_index(drop=True)
    return new_df
problem_type_to_misconceptions_mapping={p:set() for p in final_training_set['problem_type'].unique()}
for idx, row in final_training_set.iterrows():
  problem_type=row['problem_type']
  misconception=row['Misconception']
  for cls in misconception:
    problem_type_to_misconceptions_mapping[problem_type].add(cls)
# Use the mapping you just created
final_balanced_set = balance_training_set_by_type(final_training_set, problem_type_to_misconceptions_mapping)

Problem Type: 12 | Calculated Cap (Threshold): 465
Problem Type: 5 | Calculated Cap (Threshold): 109
Problem Type: 11 | Calculated Cap (Threshold): 199
Problem Type: 13 | Calculated Cap (Threshold): 1
Problem Type: 8 | Calculated Cap (Threshold): 298
Problem Type: 3 | Calculated Cap (Threshold): 83
Problem Type: 2 | Calculated Cap (Threshold): 42
Problem Type: 9 | Calculated Cap (Threshold): 1
Problem Type: 1 | Calculated Cap (Threshold): 310
Problem Type: 4 | Calculated Cap (Threshold): 476
Problem Type: 10 | Calculated Cap (Threshold): 329
Problem Type: 14 | Calculated Cap (Threshold): 32
Problem Type: 7 | Calculated Cap (Threshold): 244
Problem Type: 15 | Calculated Cap (Threshold): 1


### **Final training/validation null counts.**

In [49]:
# Check nulls for final_training_set
print("--- Null Values in final_training_set ---")
print(final_training_set.isnull().sum())

print("\n--- Null Values in agg_val ---")
print(agg_val.isnull().sum())

--- Null Values in final_training_set ---
QuestionText                        0
StudentExplanation                  0
MC_Answer                           0
Category                            0
Misconception                       0
Correctness                         0
problem_type                        0
Structural_Key                  57369
function                        57380
Original_category               58015
row_id                          55382
tuple_key                       55382
fname                           59175
Mathematical_Context                0
Original_Explanation                0
Mathematical_Context_Reduced        0
temp_key                            0
dtype: int64

--- Null Values in agg_val ---
QuestionText                        0
StudentExplanation                  0
MC_Answer                           0
Category                            0
Misconception                       0
Correctness                         0
row_id                          76717
p

In [50]:
# --- DEBUGGING BLOCK ---

print("\n--- DEBUGGING DATA PROPERTIES BEFORE TRAINING ---\n")

# 1. Overall Sizes (The most critical check)
print(f"Total training samples (final_training_set): {len(final_training_set)}")
print(f"Total validation samples (agg_val): {len(agg_val)}")

if len(agg_val) == 0:
    raise ValueError("FATAL ERROR: The validation dataset (agg_val) is empty, causing the IndexError.")

# 2. Check the first few rows of the validation set
print("\nFirst 5 rows of Validation Set (agg_val):")
print(agg_val.head())

# 3. Check for Null/Missing Data (Could cause downstream issues if not handled)
print("\nValidation Set (agg_val) Null/Missing Counts:")
print(agg_val.isnull().sum())

# 4. Check Key Column Data Types
print("\nValidation Set (agg_val) Column Types:")
print(agg_val[['problem_type', 'Misconception', 'Category', 'Correctness']].dtypes)


# 5. Check Distribution of Key Categorical Columns (Reveals filtering errors)
print("\nValidation Set (agg_val) MISCONCEPTION Distribution (Top 10):")
# Use .str.join(',') if 'Misconception' is a list/object column, otherwise use .value_counts()
if agg_val['Misconception'].apply(type).iloc[0] == list:
    print(agg_val['Misconception'].apply(lambda x: x[0] if x else 'UNKNOWN').value_counts().head(10))
else:
    print(agg_val['Misconception'].value_counts().head(10))

print("\nValidation Set (agg_val) CATEGORY Distribution (True/False):")
# Assuming Category is a list like ['True_Misconception']
print(agg_val['Category'].apply(lambda x: x[0].split('_')[0] if isinstance(x, list) and x else 'UNKNOWN').value_counts())

print("\nValidation Set (agg_val) CORRECTNESS Distribution:")
print(agg_val['Correctness'].value_counts())

print("\n-------------------------------------------------\n")

# --- END DEBUGGING BLOCK ---


--- DEBUGGING DATA PROPERTIES BEFORE TRAINING ---

Total training samples (final_training_set): 59821
Total validation samples (agg_val): 79691

First 5 rows of Validation Set (agg_val):
                                         QuestionText  \
4   A bag contains \( 24 \) yellow and green balls...   
7   A bag contains \( 24 \) yellow and green balls...   
14  A bag contains \( 24 \) yellow and green balls...   
18  A bag contains \( 24 \) yellow and green balls...   
19  A bag contains \( 24 \) yellow and green balls...   

                                   StudentExplanation MC_Answer  \
4   FRAC_COUNT / FRACTION_DENOMINATOR of ALL_UNITS...   \( 8 \)   
7   FRAC_COUNT 8th of ALL_UNITS is WHOLE_DIV_DEN_S...  \( 15 \)   
14  FRAC_COUNT/FRACTION_DENOMINATOR equals WHOLE_D...   \( 3 \)   
18  FRAC_COUNT/FRACTION_DENOMINATOR is WHOLE_DIV_D...  \( 15 \)   
19  FRAC_COUNT/FRACTION_DENOMINATOR is WHOLE_DIV_D...  \( 15 \)   

           Category       Misconception  Correctness   row_id  pro

In [51]:
## 💾 Problem Type Sample Filter

def print_one_sample_per_problem_type(df):
    """
    Filters the DataFrame for one unique sample of each ProblemType
    and prints its key features clearly.
    """
    # Use drop_duplicates to efficiently get one row per ProblemType
    samples = df.drop_duplicates(subset=['problem_type'], keep='first')

    print("=" * 70)
    print("🎯 ONE SAMPLE PER PROBLEM TYPE (POST-FEATURE REDUCTION)")
    print("=" * 70)

    # Iterate through the selected samples and print the required details
    for index, row in samples.iterrows():
        print(f"\n--- 📁 PROBLEM TYPE: **{row['problem_type']}** ---")
        print(f"**QuestionText:** {row['QuestionText']}")
        print(f"**MC_Answer:** {row['MC_Answer']}")
        print(f"**Misconception:** {row['Misconception']}")
        print(f"**Mathematical_Context:**\n{row['Mathematical_Context']}")
        print("-" * 30)

# Run the function on your training data
print_one_sample_per_problem_type(final_training_set)

🎯 ONE SAMPLE PER PROBLEM TYPE (POST-FEATURE REDUCTION)

--- 📁 PROBLEM TYPE: **12** ---
**QuestionText:** Calculate (-13)-(-18)
**MC_Answer:** \( 5 \)
**Misconception:** ['No Misconception']
**Mathematical_Context:**
 | SA:5 | NUM_COUNT:2 | FRAC_COUNT:0 | N1:-13 | N2:-18 | N1+N2:-31 | N1-N2:5 | N1xN2:234 | N1divN2:0.722222 | N2divN1:1.38462 | CORRECT_ANSWER:5 | N1_ABS:13 | N1_OPP:13 | N1_N2_ABS_SUM:31 | N1_N2_ABS_DIFF:5 | N2_OPP:18 | Possible_Tacking_Answer: -31 | N2_ABS:18 | N2_OPP:18 | ANSWER_HAS_FRACTION:FALSE |
------------------------------

--- 📁 PROBLEM TYPE: **5** ---
**QuestionText:** \( \frac{3}{12} + \frac{5}{9} = \)
**MC_Answer:** \( \frac{8}{21} \)
**Misconception:** ['Adding_across']
**Mathematical_Context:**
 | SA:8/21 | NUM_COUNT:4 | FRAC_COUNT:2 | N1:3 | N2:5 | N3:9 | N4:12 | F1:1/4 | F1_NUM:1 | F1_DEN:4 | F1_ORIG_NUM:3 | F1_ORIG_DEN:12 | F1_NUM_LT_DEN:TRUE | F1_RECIP:4/1 | F2:5/9 | F2_NUM:5 | F2_DEN:9 | F2_ORIG_NUM:5 | F2_ORIG_DEN:9 | F2_NUM_LT_DEN:TRUE | F2_RECIP:9/5 

## 🧠 Multi-Task Model Architecture and Contrastive Batching

This code block defines the final training components: the **Multi-Task Qwen Model**, the custom **Dataset**, and the strategic **Contrastive Sampler**.

---

### 1. 🤖 MultiTaskQwen Model Architecture

The `MultiTaskQwen` class implements the architecture for solving the problem using a **Qwen2-7B-Instruct** foundation model via the **LoRA** (Low-Rank Adaptation) technique for parameter-efficient fine-tuning.

* **Foundation Model:** Uses `AutoModel.from_pretrained` to load Qwen (likely with **BitsAndBytes** 4-bit quantization, though the config is passed externally).
* **Input:** The model accepts `input_ids` and `attention_mask` tensors, which contain the concatenated problem text, mathematical context, and student explanation.
* **Text Representation:** The forward pass extracts the hidden state of the **last non-padding token** (the `text_representation`). This single vector summarizes the entire input sequence.
* **Multi-Task Heads (Classification):** Three separate linear layers (heads) are attached to this final representation, allowing the model to perform three distinct tasks simultaneously:
    * `self.correctness_head`: Binary classification (Correct vs. Incorrect).
    * `self.category_head`: Multi-label classification for the **Category** (e.g., `True_Misconception`, `False_Correct`).
    * `self.misconception_head`: Multi-label classification for the **Misconception** (37+ classes).
* **Output:** The model returns three sets of **logits** (raw prediction scores), one for each task head.

---

### 2. 📚 MultiTaskDatasetTextAugmentedB

This custom `Dataset` class prepares the final input for the model.

* **Input Formatting:** It combines the `QuestionText`, the cleaned/augmented `StudentExplanation`, and the engineered `Mathematical_Context` features into a single input string for the tokenizer.
    * The `Mathematical_Context` is specifically formatted into a structured text block: `<MATH_CONTEXT> SA:X | N1:Y | ... </MATH_CONTEXT>`.
* **Text Augmentation:** It applies the `augmenter.augment()` function on the fly to the explanation text, introducing typos, synonyms, and random capitalization to increase the model's robustness to noisy student input.
* **Multi-Label Encoding:** It converts the list of `Category` and `Misconception` strings into **multi-hot encoded vectors** (`torch.float32`), which is required for the Binary Cross-Entropy Loss used in the multi-label tasks.

---

### 3. 🎲 ProblemTypeContrastiveSampler

This custom sampler is designed to create **strategic, highly informative batches** for training. Its goal is to maximize the learning signal within each step by forcing the model to make fine-grained distinctions.

* **Contrastive Learning Objective:** The sampler prioritizes batches where the **Problem Type is held constant**, but the **Misconception is varied**.
    * **Goal:** If a batch contains three fraction division problems, but one has the 'Inversion' misconception, one has 'Multiplication', and one is 'Correct', the model is forced to focus on the subtle linguistic and numerical differences that distinguish these outcomes *within the same problem structure*.
* **Sampling Strategy:**
    1.  Select an **anchor** sample randomly.
    2.  Fill 4-5 slots with samples from the **SAME** `problem_type` but with **DIFFERENT** `Misconception` labels.
    3.  Fill 2 slots with samples having the **SAME** anchor `Misconception` but from **DIFFERENT** `problem_type`s.
    4.  Fill the rest with nearby random samples.
* **Benefit:** This approach accelerates learning by mitigating the bias of the problem type (e.g., all fraction problems look similar) and forces the model to heavily rely on the **explanation text** and **mathematical context features** to classify the specific error.

In [29]:
# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================
# Improved evaluation for multi-label (assuming your provided version is available)
def evaluate_multilabel_comprehensive(y_true, y_pred, label_names=None):
    # ... (Your existing evaluate_multilabel_comprehensive function body)
    results = {}

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    results['micro_f1'] = f1_score(y_true, y_pred, average='micro', zero_division=0)
    results['macro_f1'] = f1_score(y_true, y_pred, average='macro', zero_division=0)
    results['weighted_f1'] = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    results['samples_f1'] = f1_score(y_true, y_pred, average='samples', zero_division=0)

    results['hamming_loss'] = np.mean(y_true != y_pred)
    results['subset_accuracy'] = np.mean(np.all(y_true == y_pred, axis=1))

    return results
def calculate_class_weights(df, label_encoder, column_name='Category'):
    """
    Calculate class weights to handle imbalanced categories.
    Works for both single-label and multi-label scenarios.

    Args:
        df: DataFrame containing the data
        label_encoder: LabelEncoder with fitted classes
        column_name: Name of the column containing labels (default: 'Category')

    Returns:
        torch.Tensor: Class weights for use in loss functions
    """
    all_items = []

    for items in df[column_name]:
        if isinstance(items, list):
            # Multi-label case: extend the list
            all_items.extend(items)
        else:
            # Single-label case: append the item
            all_items.append(items)

    # Count occurrences of each class
    item_counts = Counter(all_items)

    # Calculate weights using inverse frequency
    total_items = len(all_items)
    num_classes = len(label_encoder.classes_)

    weights = []
    for class_name in label_encoder.classes_:
        count = item_counts.get(class_name, 1)  # Use 1 as minimum to avoid division by zero
        # Standard formula: total / (num_classes * count)
        weight = total_items / (num_classes * count)
        weights.append(weight)

    return torch.tensor(weights, dtype=torch.float32)


# ============================================================================
# MODEL CLASS
# ============================================================================

class MultiTaskQwen(nn.Module):
    def __init__(self, pretrained_model_name, num_categories, num_misc_classes, bnb_config=None):
        """
        Multi-task model using Qwen and fine-tuning the classification heads.

        Args:
            pretrained_model_name: Name of pretrained transformer model (e.g., 'Qwen/Qwen2-7B-Instruct')
            num_categories: Number of category classes
            num_misc_classes: Number of misconception classes
            bnb_config: BitsAndBytesConfig for 4-bit loading
        """
        super(MultiTaskQwen, self).__init__()

        # Load Qwen as AutoModel
        self.qwen = AutoModel.from_pretrained(
            pretrained_model_name,
            quantization_config=bnb_config,
            trust_remote_code=True
        )
        hidden_size = self.qwen.config.hidden_size

        # Task-specific heads
        self.correctness_head = nn.Linear(hidden_size, 2)
        self.category_head = nn.Linear(hidden_size, num_categories)
        self.misconception_head = nn.Linear(hidden_size, num_misc_classes)

    def forward(self, input_ids, attention_mask):
        """
        Forward pass through the model.
        """
        # Process text with Qwen
        outputs = self.qwen(input_ids=input_ids, attention_mask=attention_mask)

        # For CausalLMs (like Qwen), extract the last non-padding token's hidden state
        last_hidden_state = outputs.last_hidden_state

        # Find the last token index that is NOT padding
        sequence_lengths = torch.sum(attention_mask, dim=1) - 1

        # Extract the hidden state for the effective last token
        text_representation = last_hidden_state[
            torch.arange(last_hidden_state.size(0), device=input_ids.device),
            sequence_lengths
        ]

        # Task-specific predictions
        correctness_logits = self.correctness_head(text_representation)
        category_logits = self.category_head(text_representation)
        misconception_logits = self.misconception_head(text_representation)

        return correctness_logits, category_logits, misconception_logits


# ============================================================================
# DATASET CLASS
# ============================================================================
class MultiTaskDatasetTextAugmentedB(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, label_encoder_category, label_encoder_misconception,
                 augmenter, max_length=450, use_answer=True, use_math_context=True,
                 kind='train', task='misconception'):
        self.task = task
        self.kind = kind
        self.label_encoder_category = label_encoder_category
        self.label_encoder_misconception = label_encoder_misconception
        self.questions = df['QuestionText'].tolist()
        self.explanations = df['StudentExplanation'].tolist()
        self.correctness = df['Correctness'].tolist()
        self.mc_answer = df['MC_Answer'].tolist()
        self.categories = df['Category'].tolist()
        self.misconceptions = df['Misconception'].tolist()
        self.math_context = df['Mathematical_Context'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.use_answer = use_answer
        self.use_math_context = use_math_context
        self.augmenter = augmenter

    def __len__(self):
        return len(self.questions)

    def format_math_context_as_text(self, context_string):
        """
        Refined for Correctness: If the task is 'correctness', we keep the full
        feature list. Otherwise, we use the cleaned key:value format.
        """
        if not context_string or not self.use_math_context:
            return ""

        if self.task == 'correctness':
            # For correctness, we show the full list of informational features
            return f" <MATH_CONTEXT> {context_string.strip()} </MATH_CONTEXT> "

        # Standard cleaning for misconception task
        parts = [part.strip() for part in context_string.split('|') if part.strip()]
        formatted_features = []
        for part in parts:
            if ':' in part:
                key, value = part.split(':', 1)
                formatted_features.append(f"{key.strip()}:{value.strip()}")

        if not formatted_features:
            return ""

        return " <MATH_CONTEXT> " + " | ".join(formatted_features) + " </MATH_CONTEXT> "

    def __getitem__(self, idx):
        question = self.questions[idx]
        explanation = self.explanations[idx]

        if self.kind == 'train' and self.augmenter:
            explanation = self.augmenter.augment(explanation)

        correct_label = self.correctness[idx]
        category_labels = self.categories[idx]
        mc_answer = str(self.mc_answer[idx])

        # Randomly drop MC Answer during training for robustness
        if self.kind == 'train' and random.random() < 0.5 and self.task=='misconception':
            mc_answer = " "

        misconception_label = self.misconceptions[idx]
        math_context_str = self.math_context[idx]

        # Format Math Context based on the task
        math_context_text = self.format_math_context_as_text(math_context_str)

        # Logic for Task-Specific Input Formatting
        if self.task == 'correctness':
            # SHOW: Question + All Math Features | PREDICT: MC_Answer
            question_with_context = f"{question} {math_context_text}"
            explanation = mc_answer
        else:
            # SHOW: Question + Answer | PREDICT: Math Features + Explanation
            question_with_context = f"{question} {mc_answer}"
            if self.use_math_context:
                explanation = f"{math_context_text} {explanation}"

        # Category Encoding (Single-Label)
        if isinstance(category_labels, list):
            category_labels = category_labels[0]
        category_label = self.label_encoder_category.transform([category_labels])[0]

        # Misconception Encoding (Multi-Label)
        misconception_multi_label = [0] * len(self.label_encoder_misconception.classes_)
        if isinstance(misconception_label, str):
            misconception_label = [misconception_label]
        for mis in misconception_label:
            if mis in self.label_encoder_misconception.classes_:
                idx_mis = self.label_encoder_misconception.transform([mis])[0]
                misconception_multi_label[idx_mis] = 1

        # Tokenize text
        encoding = self.tokenizer(
            question_with_context,
            explanation,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'correctness': torch.tensor(correct_label, dtype=torch.long),
            'category': torch.tensor(category_label, dtype=torch.long),
            'misconception': torch.tensor(misconception_multi_label, dtype=torch.float32),
            'question_text': question_with_context,
            'explanation_text': explanation,
            'task': self.task
        }
class MultiTaskDatasetTextAugmentedB(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, label_encoder_category, label_encoder_misconception,
                 augmenter, max_length=450, use_answer=True, use_math_context=True,
                 kind='train', task='misconception'):
        self.task = task
        self.kind = kind
        self.label_encoder_category = label_encoder_category
        self.label_encoder_misconception = label_encoder_misconception
        self.questions = df['QuestionText'].tolist()
        self.explanations = df['StudentExplanation'].tolist()
        self.correctness = df['Correctness'].tolist()
        self.mc_answer = df['MC_Answer'].tolist()
        self.categories = df['Category'].tolist()
        self.misconceptions = df['Misconception'].tolist()
        self.math_context = df['Mathematical_Context'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.use_answer = use_answer
        self.use_math_context = use_math_context
        self.augmenter = augmenter

    def __len__(self):
        return len(self.questions)

    def format_math_context_as_text(self, context_string):
        """Convert math context to text format"""
        if not context_string or not self.use_math_context:
            return ""

        parts = [part.strip() for part in context_string.split('|') if part.strip()]
        formatted_features = []
        for part in parts:
            if ':' in part:
                key, value = part.split(':', 1)
                formatted_features.append(f"{key.strip()}:{value.strip()}")

        if not formatted_features:
            return ""

        return " <MATH_CONTEXT> " + " | ".join(formatted_features) + " </MATH_CONTEXT> "

    def __getitem__(self, idx):
        question = self.questions[idx]
        explanation = self.explanations[idx]

        if self.kind == 'train':
            explanation = self.augmenter.augment(explanation)

        correct_label = self.correctness[idx]
        category_labels = self.categories[idx]
        mc_answer = str(self.mc_answer[idx])

        # Randomly drop MC Answer during training for robustness (misconception only)
        if self.kind == 'train' and random.random() < 0.5 and self.task == 'misconception':
            mc_answer = " "
        if self.kind=='train' and random.random()<0.5 and self.task=='correctness':
          explanation=" "
        misconception_label = self.misconceptions[idx]
        math_context_str = self.math_context[idx]

        # Format Math Context as Text
        math_context_text = self.format_math_context_as_text(math_context_str)

        # ========== MATCH INFERENCE FORMAT ==========
        # Both tasks use: Question + MC_Answer as input 1
        question_with_context = question + " " + mc_answer

        # Both tasks use: Math_Context + Explanation as input 2
        if self.use_math_context:
            explanation = math_context_text + " " + explanation

        # Note: For correctness task, make sure df has full Mathematical_Context
        # For misconception task, make sure df has Mathematical_Context_Reduced

        # Encode category label
        if isinstance(category_labels, list):
            category_labels = category_labels[0]
        category_label = self.label_encoder_category.transform([category_labels])[0]

        # Encode misconception labels (Multi-Label)
        misconception_multi_label = [0] * len(self.label_encoder_misconception.classes_)
        if isinstance(misconception_label, str):
            misconception_label = [misconception_label]
        for mis in misconception_label:
            if mis in self.label_encoder_misconception.classes_:
                idx_mis = self.label_encoder_misconception.transform([mis])[0]
                misconception_multi_label[idx_mis] = 1

        # Tokenize text
        encoding = self.tokenizer(
            question_with_context,  # Question + MC_Answer
            explanation,            # Math_Context + Explanation
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'correctness': torch.tensor(correct_label, dtype=torch.long),
            'category': torch.tensor(category_label, dtype=torch.long),
            'misconception': torch.tensor(misconception_multi_label, dtype=torch.float32),
            'question_text': question_with_context,
            'explanation_text': explanation,
            'task': self.task
        }
class ProblemTypeContrastiveSampler:
    """
    Creates batches that leverage the problem_type column for strategic sampling.

    Each batch aims to contain:
    - Multiple examples from the SAME problem_type with DIFFERENT misconceptions
    - This helps the model learn to distinguish misconceptions within the same problem structure
    """

    def __init__(self, dataset, batch_size=8, problem_type_col='problem_type',
                 misconception_col='Misconception'):
        """
        Args:
            dataset: pandas DataFrame with the training data
            batch_size: number of samples per batch
            problem_type_col: name of column containing problem type (1-14)
            misconception_col: name of column containing misconception labels
        """
        self.dataset = dataset
        self.batch_size = batch_size
        self.problem_type_col = problem_type_col
        self.misconception_col = misconception_col

        # Group indices by problem type
        self.problem_type_groups = self._group_by_problem_type()

        # Within each problem type, group by misconception
        self.problem_misc_groups = self._group_by_problem_and_misconception()

    def _group_by_problem_type(self):
        """Group dataset indices by problem_type"""
        groups = {}

        for idx, row in self.dataset.iterrows():
            problem_type = row[self.problem_type_col]

            if problem_type not in groups:
                groups[problem_type] = []
            groups[problem_type].append(idx)

        return groups

    def _group_by_problem_and_misconception(self):
        """Group indices by (problem_type, misconception) pairs"""
        groups = {}

        for idx, row in self.dataset.iterrows():
            problem_type = row[self.problem_type_col]
            misconceptions = row[self.misconception_col]

            # Handle both list and single string misconceptions
            if isinstance(misconceptions, str):
                misconceptions = [misconceptions]

            for misc in misconceptions:
                key = (problem_type, misc)
                if key not in groups:
                    groups[key] = []
                groups[key].append(idx)

        return groups

    def __len__(self):
        """Returns the total number of batches"""
        return math.ceil(len(self.dataset) / self.batch_size)

    def __iter__(self):
        """Yield batches with strategic diversity within same problem types"""

        all_indices = list(range(len(self.dataset)))
        random.shuffle(all_indices)

        for i in range(0, len(all_indices), self.batch_size):
            batch_indices = []

            # Start with random anchor
            anchor_idx = all_indices[i]
            batch_indices.append(anchor_idx)

            anchor_row = self.dataset.iloc[anchor_idx]
            anchor_problem_type = anchor_row[self.problem_type_col]
            anchor_misc = anchor_row[self.misconception_col]

            # Handle anchor misconception (could be list or string)
            if isinstance(anchor_misc, list):
                anchor_misc = anchor_misc[0] if anchor_misc else None

            # Strategy 1: Add 4-5 examples from SAME problem type with DIFFERENT misconceptions
            if anchor_problem_type in self.problem_type_groups:
                same_problem_type = [
                    idx for idx in self.problem_type_groups[anchor_problem_type]
                    if idx != anchor_idx
                ]

                # Try to get diverse misconceptions within same problem type
                diverse_indices = self._get_diverse_misconceptions(
                    same_problem_type,
                    anchor_misc,
                    target_count=min(5, len(same_problem_type))
                )
                batch_indices.extend(diverse_indices)

            # Strategy 2: Add 2 examples with SAME misconception but from DIFFERENT problem types
            if anchor_misc and (anchor_problem_type, anchor_misc) in self.problem_misc_groups:
                # Get all indices with same misconception
                same_misc_indices = []
                for (pt, misc), indices in self.problem_misc_groups.items():
                    if misc == anchor_misc and pt != anchor_problem_type:
                        same_misc_indices.extend(indices)

                if same_misc_indices:
                    batch_indices.extend(
                        random.sample(same_misc_indices, min(2, len(same_misc_indices)))
                    )

            # Fill remaining slots with random samples from the current window
            remaining = self.batch_size - len(batch_indices)
            if remaining > 0:
                available = [
                    idx for idx in all_indices[i:i+self.batch_size]
                    if idx not in batch_indices
                ]
                batch_indices.extend(available[:remaining])

            # Pad if batch is still too small
            while len(batch_indices) < self.batch_size:
                batch_indices.append(random.choice(all_indices))

            # Ensure we only yield exactly batch_size items
            yield batch_indices[:self.batch_size]

    def _get_diverse_misconceptions(self, candidate_indices, exclude_misc, target_count):
        """
        Select samples with diverse misconceptions from candidates.

        Args:
            candidate_indices: list of dataset indices to choose from
            exclude_misc: misconception to avoid (from anchor)
            target_count: number of samples to select

        Returns:
            list of selected indices with diverse misconceptions
        """
        # Group candidates by their misconceptions
        misc_buckets = {}

        for idx in candidate_indices:
            row_misc = self.dataset.iloc[idx][self.misconception_col]

            # Handle list or string
            if isinstance(row_misc, list):
                row_misc = row_misc[0] if row_misc else "Unknown"

            # Skip if same as anchor misconception
            if row_misc == exclude_misc:
                continue

            if row_misc not in misc_buckets:
                misc_buckets[row_misc] = []
            misc_buckets[row_misc].append(idx)

        # Sample one from each misconception bucket (for diversity)
        selected = []
        for misc, indices in misc_buckets.items():
            if len(selected) >= target_count:
                break
            selected.append(random.choice(indices))

        # If we need more, randomly sample from remaining candidates
        if len(selected) < target_count:
            remaining = [idx for idx in candidate_indices if idx not in selected]
            selected.extend(
                random.sample(remaining, min(target_count - len(selected), len(remaining)))
            )

        return selected


# ============================================================================
# USAGE EXAMPLE
# ============================================================================

# Create the sampler
contrastive_sampler = ProblemTypeContrastiveSampler(
    final_training_set,
    batch_size=16,
    problem_type_col='problem_type',
    misconception_col='Misconception'
)



## 🚀 QLoRA Multi-Task Training Loop Explained

This code defines the **`continue_training_with_misconception_qwen`** function, which orchestrates the entire training process for your specialized Qwen-based multi-task model. This function brings together the custom model, dataset, sampler, and loss calculation to fine-tune the model for the three classification tasks.

---

### 1. ⚙️ Setup and Configuration

* **Model Loading (QLoRA):** The function loads the **Qwen2-7B-Instruct** model using **BitsAndBytes (BNB)** configuration to perform **4-bit quantization (QLoRA)**. This dramatically reduces memory usage, allowing a large 7B parameter model to be fine-tuned on consumer GPUs.
* **LoRA Injection:** The model is prepared for k-bit training, and **LoRA adapters** are injected into the transformer's attention layers (`q_proj`, `v_proj`, `k_proj`, `o_proj`). This means only the small, low-rank adapters are actually trained, keeping the massive Qwen parameters frozen.
* **Optimizer:** The **AdamW8bit** optimizer from the `bitsandbytes` library is used, which is optimized for training models with 8-bit quantized weights.
* **Data Loaders:**
    * The **`MultiTaskDatasetTextAugmentedB`** is used, providing the combined text input and multi-hot encoded labels.
    * The training data uses the **`ProblemTypeContrastiveSampler`** for strategic batching, which ensures each batch maximizes the distinction between different misconceptions within the same problem type.

---

### 2. ⚖️ Loss Function and Weighting

The model is trained simultaneously on three distinct tasks, each with its own loss function:

| Task | Loss Function | Weighting | Purpose |
| :--- | :--- | :--- | :--- |
| **Correctness** (Binary) | `nn.CrossEntropyLoss` | Weight $\times 1.0$ | Standard classification for correct/incorrect answer. |
| **Category** (Multi-Label) | `nn.BCEWithLogitsLoss` | Weight $\times 1.0$ (with **Class Weights**) | Classifies samples into high-level categories (e.g., `True_Correct`). Class weights from `calculate_class_weights` handle class imbalance. |
| **Misconception** (Multi-Label) | `nn.BCEWithLogitsLoss` | Weight $\times **2.5**$ (with **Class Weights**) | Classifies the specific fine-grained misconceptions. The loss is given a **higher weight (2.5)** to prioritize the difficult and critical task of identifying the specific error over the other two tasks. |

The **Total Loss** is the sum of these three weighted losses, and this is the value the model attempts to minimize.

---

### 3. 🔁 Training and Evaluation

* **Gradient Accumulation:** Training uses a large effective batch size (`BATCH_SIZE` $\times$ `GRADIENT_ACCUMULATION_STEPS` $= 16 \times 4 = 64$) by accumulating gradients over multiple steps before performing a single `optimizer.step()`.
* **Checkpointing:** The function supports **resuming training** from a saved checkpoint, which is essential for long-running fine-tuning jobs.
* **Model Saving:** The model saves the best checkpoint based on a combination of **Validation Loss** and the **average F1 score** across the three tasks (`avg_val_f1`).
* **Early Stopping:** A `patience` mechanism is implemented to halt training if the validation performance does not improve after a set number of epochs, preventing overfitting and saving compute time.

In [30]:
# ============================================================================
# GLOBAL FLAG: Set to True to train only on misconceptions
# ============================================================================
MISCONCEPTION_ONLY = True
if MISCONCEPTION_ONLY:
    final_training_set['Mathematical_Context']=final_training_set['Mathematical_Context_Reduced']
    agg_val['Mathematical_Context']=agg_val['Mathematical_Context_Reduced']
    print("🎯 Training in MISCONCEPTION_ONLY mode")
else:
    print("🎯 Training in MULTI-TASK mode")
# runtime.disconnect_runtime()

# ============================================================================
# TRAINING FUNCTION - WITH MISCONCEPTION_ONLY TOGGLE
# ============================================================================

def continue_training_with_misconception_qwen(
    final_training_set,
    agg_val,
    label_encoder_category,
    label_encoder_misconception,
    augmenter,
    drive_dir,
    evaluate_multilabel_comprehensive,
    checkpoint_path=None,
    original_transfer=False,
    task='misconception'
):
    """
    Main training function for the Qwen multi-task model.

    When MISCONCEPTION_ONLY=True: Only trains on misconception classification
    When MISCONCEPTION_ONLY=False: Trains on all three tasks (correctness, category, misconception)

    Args:
        final_training_set: Training DataFrame
        agg_val: Validation DataFrame
        label_encoder_category: Fitted LabelEncoder for categories
        label_encoder_misconception: Fitted LabelEncoder for misconceptions
        augmenter: Text augmenter object
        drive_dir: Directory to save checkpoints
        evaluate_multilabel_comprehensive: Function to evaluate multi-label metrics
        checkpoint_path: Path to checkpoint to resume from (optional)
        original_transfer: If True, transfer from Stage 1; if False, resume Stage 2 training

    Returns:
        Trained model
    """
    # --- Configuration ---
    MODEL_NAME = 'Qwen/Qwen2-7B-Instruct'
    LORA_R = 64
    LORA_ALPHA = 64
    LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]
    BATCH_SIZE = 16
    NUM_EPOCHS = 2
    GRADIENT_ACCUMULATION_STEPS = 4

    # Loss weights
    if MISCONCEPTION_ONLY:
        MISCONCEPTION_LOSS_WEIGHT = 2.5
        CATEGORY_LOSS_WEIGHT = 0.0
        CORRECTNESS_LOSS_WEIGHT = 0.0
        print("🎯 Training in MISCONCEPTION_ONLY mode")
    else:
        MISCONCEPTION_LOSS_WEIGHT = 0
        CATEGORY_LOSS_WEIGHT = 0.0
        CORRECTNESS_LOSS_WEIGHT = 2.5
        print("🎯 Training in MULTI-TASK mode")

    # --- Tokenizer Setup ---
    from transformers import AutoTokenizer, BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.model_max_length = 450

    # --- Model Loading ---
    print(f"🔄 Loading model: {MODEL_NAME} with 4-bit QLoRA...")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True
    )

    model = MultiTaskQwen(
        MODEL_NAME,
        len(label_encoder_category.classes_),
        len(label_encoder_misconception.classes_),
        bnb_config=bnb_config
    )

    # LoRA Preparation
    model.qwen = prepare_model_for_kbit_training(model.qwen)

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=0.07,
        bias="none",
        task_type="FEATURE_EXTRACTION"
    )

    model.qwen = get_peft_model(model.qwen, lora_config)
    print("✅ LoRA adapters injected into the model's transformer backbone.")

    try:
        model.qwen.print_trainable_parameters()
    except AttributeError:
        print("Trainable parameters information not available.")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    start_epoch = 0
    best_val_loss = float('inf')
    best_val_f1 = 0.0

    # --- Optimizer ---
    optimizer = AdamW8bit(model.parameters(), lr=9e-6, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

    # --- Checkpoint Loading ---
    if original_transfer:
        # Transfer learning from Stage 1 (no category head)
        if checkpoint_path is not None and os.path.exists(checkpoint_path):
            try:
                checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

                # Get the saved state dict
                saved_state_dict = checkpoint['model_state_dict']

                # Get current model's state dict
                current_state_dict = model.state_dict()

                # Filter out keys that don't match in size or don't exist
                compatible_state_dict = {}
                incompatible_keys = []

                for key, value in saved_state_dict.items():
                    if key in current_state_dict:
                        # Check if shapes match
                        if current_state_dict[key].shape == value.shape:
                            compatible_state_dict[key] = value
                        else:
                            incompatible_keys.append(f"{key}: saved={value.shape}, current={current_state_dict[key].shape}")
                    else:
                        incompatible_keys.append(f"{key}: not found in current model")

                # Load only compatible weights
                model.load_state_dict(compatible_state_dict, strict=False)

                # Print what was loaded and what wasn't
                print(f"✅ Loaded {len(compatible_state_dict)} compatible parameters from checkpoint")
                if incompatible_keys:
                    print(f"⚠️ Skipped {len(incompatible_keys)} incompatible parameters:")
                    for key in incompatible_keys[:5]:  # Show first 5
                        print(f"   - {key}")
                    if len(incompatible_keys) > 5:
                        print(f"   ... and {len(incompatible_keys) - 5} more")

                print("ℹ️ Starting optimizer from scratch (new layers added)")

                start_epoch = 0
                best_val_loss = float('inf')
                best_val_f1 = 0.0

                print(f"✅ Loaded Stage 1 weights. Training Stage 2 from epoch 0 with transferred knowledge.")

            except Exception as e:
                print(f"🛑 Error loading checkpoint at {checkpoint_path}: {e}")
                print("Starting training from scratch instead.")
        else:
            print("No valid checkpoint found - starting training from scratch")
    else:
        # Resume Stage 2 training (all heads match)
        if checkpoint_path is not None and os.path.exists(checkpoint_path):
            try:
                checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
                model.load_state_dict(checkpoint['model_state_dict'], strict=False)
                optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
                start_epoch = checkpoint.get('epoch', 0) + 1
                best_val_loss = checkpoint.get('val_loss', float('inf'))
                best_val_f1 = checkpoint.get('val_f1', 0.0)

                print(f"✅ Loaded checkpoint from {checkpoint_path}. Resuming from epoch {start_epoch}.")
                print(f"Best Validation Loss: {best_val_loss}")
                print(f"Best Validation F1: {best_val_f1}")
            except Exception as e:
                print(f"🛑 Error loading checkpoint at {checkpoint_path}: {e}")
                print("Starting training from scratch instead.")
        else:
            print("No valid checkpoint found - starting training from scratch")
    #best_val_loss = float('inf')
    #best_val_f1 = 0.0
    # --- Data Loading ---
    contrastive_sampler = ProblemTypeContrastiveSampler(
        final_training_set,
        batch_size=16,
        problem_type_col='problem_type',
        misconception_col='Misconception'
    )

    train_dataset = MultiTaskDatasetTextAugmentedB(
        final_training_set, tokenizer, label_encoder_category, label_encoder_misconception, augmenter,
        max_length=tokenizer.model_max_length,
        use_math_context=True,
        kind='train',
        task=task
    )

    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_sampler=contrastive_sampler
    )

    val_dataset = MultiTaskDatasetTextAugmentedB(
        agg_val, tokenizer, label_encoder_category, label_encoder_misconception, augmenter,
        max_length=tokenizer.model_max_length,
        use_math_context=True,
        kind='val',
        task=task
    )
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # --- Loss Setup ---
    class_weights_cat = calculate_class_weights(final_training_set, label_encoder_category, column_name='Category').to(device)
    class_weights_mis = calculate_class_weights(final_training_set, label_encoder_misconception, column_name='Misconception').to(device)

    criterion_correctness = nn.CrossEntropyLoss()
    criterion_category = nn.CrossEntropyLoss(weight=class_weights_cat)
    criterion_misconception = nn.BCEWithLogitsLoss(pos_weight=class_weights_mis)

    # --- Training Loop ---
    save_dir = drive_dir
    os.makedirs(save_dir, exist_ok=True)

    patience = 4
    patience_counter = 0
    print(f"Best Val f1: {best_val_f1}")
    for epoch in range(NUM_EPOCHS):
        actual_epoch = start_epoch + epoch
        model.train()
        train_losses = []

        optimizer.zero_grad()

        for batch_idx, batch in enumerate(train_loader):
            # Move tensors to device
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            misconception_labels = batch['misconception'].to(device)

            # Forward pass
            logits_correctness, logits_category, logits_misconception = model(
                input_ids, attention_mask
            )

            # Loss calculation
            loss_misconception = criterion_misconception(logits_misconception, misconception_labels)

            if MISCONCEPTION_ONLY:
                total_loss = loss_misconception * MISCONCEPTION_LOSS_WEIGHT
            else:
                correctness_labels = batch['correctness'].to(device)
                category_labels = batch['category'].to(device)
                loss_correctness = criterion_correctness(logits_correctness, correctness_labels)
                loss_category = criterion_category(logits_category, category_labels)
                total_loss = (
                    loss_correctness * CORRECTNESS_LOSS_WEIGHT +
                    loss_category * CATEGORY_LOSS_WEIGHT +
                    loss_misconception * MISCONCEPTION_LOSS_WEIGHT
                )

            # Backward pass (with gradient accumulation)
            (total_loss / GRADIENT_ACCUMULATION_STEPS).backward()

            train_losses.append(total_loss.item())

            # Step optimizer when accumulation is complete
            is_accumulated_step = (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0
            is_last_batch = batch_idx == len(train_loader) - 1

            if is_accumulated_step or is_last_batch:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

                if (batch_idx + 1) % 10 == 0:
                    print(f"Epoch {actual_epoch+1} Batch {batch_idx+1}/{len(train_loader)} - Loss: {np.mean(train_losses[-GRADIENT_ACCUMULATION_STEPS:]):.4f} (Effective Step)")
            elif (batch_idx + 1) % 10 == 0:
                print(f"Epoch {actual_epoch+1} Batch {batch_idx+1}/{len(train_loader)} - Loss: {total_loss.item():.4f} (Accumulating)")

        # --- Validation Loop ---
        model.eval()
        all_misconception_preds, all_misconception_labels = [], []
        if not MISCONCEPTION_ONLY:
            all_correctness_preds, all_correctness_labels = [], []
            all_category_preds, all_category_labels = [], []
        val_losses = []

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                misconception_labels = batch['misconception'].to(device)

                logits_correctness, logits_category, logits_misconception = model(
                    input_ids, attention_mask
                )

                # Calculate loss (matching training)
                loss_misconception = criterion_misconception(logits_misconception, misconception_labels)

                if MISCONCEPTION_ONLY:
                    val_loss = loss_misconception * MISCONCEPTION_LOSS_WEIGHT
                else:
                    correctness_labels = batch['correctness'].to(device)
                    category_labels = batch['category'].to(device)
                    loss_correctness = criterion_correctness(logits_correctness, correctness_labels)
                    loss_category = criterion_category(logits_category, category_labels)
                    val_loss = (
                        loss_correctness * CORRECTNESS_LOSS_WEIGHT +
                        loss_category * CATEGORY_LOSS_WEIGHT +
                        loss_misconception * MISCONCEPTION_LOSS_WEIGHT
                    )

                val_losses.append(val_loss.item())

                # Collect predictions
                preds_misconception = (torch.sigmoid(logits_misconception) > 0.5).cpu().numpy().astype(int)
                misconception_labels_np = misconception_labels.cpu().numpy()

                all_misconception_preds.extend(preds_misconception)
                all_misconception_labels.extend(misconception_labels_np)

                if not MISCONCEPTION_ONLY:
                    preds_correctness = logits_correctness.argmax(dim=-1).cpu().numpy()
                    preds_category = logits_category.argmax(dim=-1).cpu().numpy()
                    correctness_labels_np = correctness_labels.cpu().numpy()
                    category_labels_np = category_labels.cpu().numpy()

                    all_correctness_preds.extend(preds_correctness)
                    all_correctness_labels.extend(correctness_labels_np)
                    all_category_preds.extend(preds_category)
                    all_category_labels.extend(category_labels_np)

        # --- Evaluation ---
        avg_val_loss = np.mean(val_losses)
        val_misconception_f1 = f1_score(
            all_misconception_labels,
            all_misconception_preds,
            average='macro',
            zero_division=0
        )

        if MISCONCEPTION_ONLY:
            avg_val_f1 = val_misconception_f1
            print(f"Epoch {actual_epoch+1} Validation - Loss: {avg_val_loss:.4f}")
            print(f"Epoch {actual_epoch+1} Validation - Misconception F1: {val_misconception_f1:.4f}")
        else:
            val_correctness_f1 = f1_score(
                all_correctness_labels,
                all_correctness_preds,
                average='weighted',
                zero_division=0
            )
            val_category_f1 = f1_score(
                all_category_labels,
                all_category_preds,
                average='macro',
                zero_division=0
            )
            avg_val_f1 = val_correctness_f1

            print(f"Epoch {actual_epoch+1} Validation - Loss: {avg_val_loss:.4f}")
            print(f"Epoch {actual_epoch+1} Validation - Correctness F1: {val_correctness_f1:.4f}")
            print(f"Epoch {actual_epoch+1} Validation - Category Macro F1: {val_category_f1:.4f}")
            print(f"Epoch {actual_epoch+1} Validation - Misconception F1: {val_misconception_f1:.4f}")

        scheduler.step(avg_val_loss)

        try:
            # --- Save Best Model ---
            if avg_val_loss < best_val_loss or avg_val_f1 > best_val_f1:
                print(f"💾 Saving model at epoch {actual_epoch+1} with val_loss: {avg_val_loss:.4f}, val_f1: {avg_val_f1:.4f}")
                best_val_loss = min(best_val_loss, avg_val_loss)
                best_val_f1 = max(best_val_f1, avg_val_f1)

                save_filename = 'stage2_explanation_model_dec5_MISC_ONLY_SUB_Other.pt' if MISCONCEPTION_ONLY else 'stage2_explanation_model_dec5_restructure.pt'
                save_path = os.path.join(checkpoint_dir, save_filename)

                torch.save({
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'epoch': actual_epoch,
                    'val_loss': avg_val_loss,
                    'val_f1': avg_val_f1,
                    'misconception_only': MISCONCEPTION_ONLY,
                    'label_encoder_category': label_encoder_category,
                    'label_encoder_misconception': label_encoder_misconception
                }, save_path)
                patience_counter = 0
            else:
                patience_counter += 1
                # Still save checkpoint even if not best
                save_filename = 'stage2_explanation_model_dec5_MISC_ONLY_SUB_Other_B.pt' if MISCONCEPTION_ONLY else 'stage2_explanation_model_dec5_restructure_B.pt'
                save_path = os.path.join(checkpoint_dir, save_filename)
                print(f"💾 Saving checkpoint at epoch to {save_filename}")
                torch.save({
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'epoch': actual_epoch,
                    'val_loss': avg_val_loss,
                    'val_f1': avg_val_f1,
                    'misconception_only': MISCONCEPTION_ONLY,
                    'label_encoder_category': label_encoder_category,
                    'label_encoder_misconception': label_encoder_misconception
                }, save_path)

            if patience_counter >= patience:
                print(f"⏹️ Early stopping at epoch {actual_epoch+1}")
                break
        except Exception as e:
            print(f"❌ Error saving checkpoint: {e}")
            # Try to save anyway
            save_filename = 'stage2_explanation_model_dec5_MISC_ONLY_backup.pt' if MISCONCEPTION_ONLY else 'stage2_explanation_model_dec5_backup.pt'
            save_path = os.path.join(checkpoint_dir, save_filename)

            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'epoch': actual_epoch,
                'val_loss': avg_val_loss,
                'val_f1': avg_val_f1,
                'misconception_only': MISCONCEPTION_ONLY,
                'label_encoder_category': label_encoder_category,
                'label_encoder_misconception': label_encoder_misconception
            }, save_path)

    return model

# ============================================================================
# USAGE EXAMPLE
# ============================================================================

if __name__ == "__main__":
    from sklearn.model_selection import train_test_split as tts

    # Example usage (you need to provide these variables):
    # - final_training_set: Your training DataFrame
    # - agg_val: Your validation DataFrame
    # - label_encoder_category: Fitted sklearn LabelEncoder
    # - label_encoder_misconception: Fitted sklearn LabelEncoder
    # - augmenter: Your text augmentation object
    # - drive_dir: Path to save checkpoints
    # - evaluate_multilabel_comprehensive: Your evaluation function

    do_train = True  # Set to True to train
    checkpoint_path = os.path.join(checkpoint_dir, 'stage2_explanation_model_dec5_MISC_ONLY_SUB_Other.pt')
    agg_val, _ = tts(agg_val, test_size=0.5, stratify=agg_val['problem_type'], random_state=3)

    print(f"Size of agg val: {len(agg_val)}")
    print(f'Training Data size: {len(final_training_set)}')

    # Verify Category column is single-label (not list)
    print("\n=== Verifying Category Format ===")
    print(f"Category sample: {final_training_set['Category'].head(3).tolist()}")
    print(f"Category type: {type(final_training_set['Category'].iloc[0])}")

    if isinstance(final_training_set['Category'].iloc[0], list):
        print("⚠️ Converting Category from list to single values...")
        final_training_set['Category'] = final_training_set['Category'].apply(
            lambda x: x[0] if isinstance(x, list) else x
        )
        agg_val['Category'] = agg_val['Category'].apply(
            lambda x: x[0] if isinstance(x, list) else x
        )
        print("✅ Conversion complete")

    if do_train:
        trained_model = continue_training_with_misconception_qwen(
            final_training_set=final_training_set,
            agg_val=agg_val,
            label_encoder_category=label_encoder_category,
            label_encoder_misconception=label_encoder_misconception,
            augmenter=augmenter,
            drive_dir=drive_dir,
            evaluate_multilabel_comprehensive=evaluate_multilabel_comprehensive,
            checkpoint_path=checkpoint_path,
            task='misconception'
        )
        print("✅ Training completed.")

🎯 Training in MISCONCEPTION_ONLY mode
Size of agg val: 39579
Training Data size: 16002

=== Verifying Category Format ===
Category sample: ['False_Misconception', 'False_Misconception', 'True_Correct']
Category type: <class 'str'>
🎯 Training in MISCONCEPTION_ONLY mode


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

🔄 Loading model: Qwen/Qwen2-7B-Instruct with 4-bit QLoRA...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

KeyboardInterrupt: 

## **Cleaning "Other" Samples Using Model Confidence**

### The Problem
The model learned to predict misconceptions effectively but sometimes relied on **answer patterns** rather than **reasoning patterns**. For example:
- Answer: `6/10` → Predicts "Duplication"
- Even when reasoning shows: "I simplified 9/15 to 6/10" (correct method, not duplication)

The original Kaggle "Other" category (False_Neither/True_Neither) contained:
- ✅ Genuinely vague/unclear reasoning
- ❌ Mislabeled samples with clear misconceptions
- ❌ Mislabeled samples with correct reasoning

### The Solution: Model-Based Relabeling

We used the trained model's **confidence scores** to distinguish between truly ambiguous samples and mislabeled ones:

| Model Prediction | Confidence | Action | Rationale |
|-----------------|-----------|---------|-----------|
| "Other" or "Irrelevant" | Any | **Keep label** | Model agrees this is unclear |
| Specific misconception | ≥ 0.68 | **Filter out** | Clear pattern detected → was mislabeled, move to training with correct label |
| Specific misconception | < 0.68 | **Relabel as "Irrelevant"** | Model uncertain → genuinely atypical reasoning |

### Why This Matters

By keeping only **low-confidence and ambiguous samples** as "Other"/"Irrelevant" in training data, we teach the model:
- ✅ Predict specific misconceptions only when reasoning clearly shows the pattern
- ✅ Predict "Other" when reasoning is genuinely vague/unclear
- ✅ Don't over-rely on answer patterns (e.g., 6/10 doesn't always mean Duplication)

This improves **generalization to atypical explanations** and reduces false positives on edge cases.

In [53]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModel
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

def predict_misconceptions_on_other(
    other_df_agg,
    checkpoint_path,
    label_encoder_misconception,
    label_encoder_category,
    augmenter,
    top_k=2,
    batch_size=16
):
    """
    Run inference on 'Other' samples to predict actual misconception classes.

    Args:
        other_df_agg: DataFrame with 'Other' misconception samples
        checkpoint_path: Path to trained model checkpoint
        label_encoder_misconception: Fitted LabelEncoder
        label_encoder_category: Fitted LabelEncoder
        augmenter: Text augmenter
        top_k: Number of top predictions to return (default 2)
        batch_size: Batch size for inference

    Returns:
        DataFrame with added columns:
        - top_1_misconception: Top predicted class name
        - top_1_probability: Probability for top prediction
        - top_2_misconception: Second best prediction (if top_k >= 2)
        - top_2_probability: Probability for second prediction (if top_k >= 2)
    """

    MODEL_NAME = 'Qwen/Qwen2-7B-Instruct'
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print(f"🔄 Loading model from {checkpoint_path}")
    print(f"Device: {device}")

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.model_max_length = 450

    # Load model
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True
    )

    model = MultiTaskQwen(
        MODEL_NAME,
        len(label_encoder_category.classes_),
        len(label_encoder_misconception.classes_),
        bnb_config=bnb_config
    )

    # Prepare LoRA
    model.qwen = prepare_model_for_kbit_training(model.qwen)
    lora_config = LoraConfig(
        r=64,
        lora_alpha=64,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="FEATURE_EXTRACTION"
    )
    model.qwen = get_peft_model(model.qwen, lora_config)

    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'], strict=False)
    model.to(device)
    model.eval()

    print("✅ Model loaded successfully")

    # Create dataset
    print(f"📊 Processing {len(other_df_agg)} samples...")

    # Make sure Mathematical_Context_Reduced exists
    if 'Mathematical_Context_Reduced' not in other_df_agg.columns:
        print("⚠️ Creating Mathematical_Context_Reduced...")
        other_df_agg['Mathematical_Context_Reduced'] = other_df_agg['Mathematical_Context'].apply(
            filter_math_context_case_insensitive
        )
    other_df_agg['Mathematical_Context']=other_df_agg['Mathematical_Context_Reduced']
    inference_dataset = MultiTaskDatasetTextAugmentedB(
        other_df_agg,
        tokenizer,
        label_encoder_category,
        label_encoder_misconception,
        augmenter,
        max_length=450,
        use_math_context=True,
        kind='val',  # No augmentation
        task='misconception'  # Use misconception task formatting
    )

    inference_loader = DataLoader(
        inference_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    # Run inference
    all_probs = []

    print("🔮 Running inference...")
    with torch.no_grad():
        for batch_idx, batch in enumerate(inference_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            # Get predictions
            _, _, logits_misconception = model(input_ids, attention_mask)

            # Convert to probabilities
            probs = torch.sigmoid(logits_misconception).cpu().numpy()
            all_probs.append(probs)

            if (batch_idx + 1) % 50 == 0:
                print(f"  Processed {batch_idx + 1}/{len(inference_loader)} batches")

    all_probs = np.vstack(all_probs)
    print(f"✅ Inference complete. Shape: {all_probs.shape}")

    # Get top-k predictions
    results_df = other_df_agg.copy()

    for k in range(1, top_k + 1):
        # Get indices of top-k predictions
        top_k_indices = np.argsort(-all_probs, axis=1)[:, k-1]
        top_k_probs = all_probs[np.arange(len(all_probs)), top_k_indices]

        # Convert indices to class names
        top_k_classes = label_encoder_misconception.inverse_transform(top_k_indices)

        # Add to dataframe
        results_df[f'top_{k}_misconception'] = top_k_classes
        results_df[f'top_{k}_probability'] = top_k_probs

    # Add summary info
    print("\n📊 Prediction Summary:")
    print(f"Top 1 misconception distribution:")
    print(results_df['top_1_misconception'].value_counts().head(10))
    print(f"\nMean top 1 probability: {results_df['top_1_probability'].mean():.4f}")
    print(f"Median top 1 probability: {results_df['top_1_probability'].median():.4f}")

    return results_df


# ============================================================================
# USAGE
# ============================================================================
rerun=False
checkpoint_path=os.path.join(checkpoint_dir,'stage2_explanation_model_dec5_MISC_ONLY_SUB.pt')

if rerun:
  # Run prediction
  other_df_with_predictions = predict_misconceptions_on_other(
      other_df_agg=other_df_agg,
      checkpoint_path=checkpoint_path,
      label_encoder_misconception=label_encoder_misconception,
      label_encoder_category=label_encoder_category,
      augmenter=augmenter,
      top_k=2,  # Get top 2 predictions
      batch_size=16
  )

  # Save results
  output_path = os.path.join(checkpoint_dir, 'other_samples_predicted.csv')
  other_df_with_predictions.to_csv(output_path, index=False)
  print(f"\n💾 Results saved to: {output_path}")
else:
  other_df_with_predictions = pd.read_csv(os.path.join(checkpoint_dir,'other_samples_predicted.csv'))
# Show examples
print("\n📋 Sample predictions:")
sample_cols = ['QuestionText', 'Original_Explanation', 'top_1_misconception',
               'top_1_probability', 'top_2_misconception', 'top_2_probability']
print(other_df_with_predictions[sample_cols].head(10))

🔄 Loading model from /content/drive/MyDrive/LLM_Misconception_Project/checkpoints/stage2_explanation_model_dec5_MISC_ONLY_SUB.pt
Device: cuda


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2Model LOAD REPORT from: Qwen/Qwen2-7B-Instruct
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded successfully
📊 Processing 11650 samples...
⚠️ Creating Mathematical_Context_Reduced...
🔮 Running inference...
  Processed 50/729 batches
  Processed 100/729 batches
  Processed 150/729 batches
  Processed 200/729 batches
  Processed 250/729 batches
  Processed 300/729 batches
  Processed 350/729 batches
  Processed 400/729 batches
  Processed 450/729 batches
  Processed 500/729 batches
  Processed 550/729 batches
  Processed 600/729 batches
  Processed 650/729 batches
  Processed 700/729 batches
✅ Inference complete. Shape: (11650, 37)

📊 Prediction Summary:
Top 1 misconception distribution:
top_1_misconception
No Misconception           4171
Irrelevant                  872
Incomplete                  656
Other                       621
Positive                    613
Tacking                     558
Subtraction                 374
WNB                         362
Denominator-only_change     303
Wrong_term                  267
Name: count, dtype: int64

Mean top 1 probabil

In [54]:
def relabel_low_confidence_other_samples(other_df_with_predictions, threshold=0.68):
    """
    Relabel 'Other' samples and FILTER to only keep uncertain/noisy samples.

    KEEPS only:
    1. Rows where top_1_misconception is 'Other' or 'Irrelevant' (any probability)
    2. Rows where top_1_misconception is something else AND top_1_probability < 0.68

    FILTERS OUT:
    - Rows with high confidence (≥0.68) predictions of specific misconceptions
      (these will be moved to training data with their new labels)

    Final labels: Only 'Irrelevant', 'Other', 'Incomplete' or combinations
    """
    df = other_df_with_predictions.copy()

    def get_new_misconception(row):
        top_misc = row['top_1_misconception']
        prob = row['top_1_probability']
        math_context = str(row['Mathematical_Context'])

        has_unsimplified = 'ANSWER_FRAC_IS_SIMPLIFIED:False' in math_context

        # If predicted as 'Other' or 'Irrelevant', keep it
        if top_misc in ['Other', 'Irrelevant']:
            base_labels = [top_misc]

        # For other misconceptions with low confidence
        elif prob < threshold:
            base_labels = ['Irrelevant']

        # High confidence specific misconceptions - mark for filtering
        else:
            return None  # Will be filtered out

        # Add 'Incomplete' flag if answer not simplified
        if has_unsimplified:
            base_labels.append('Incomplete')

        return base_labels

    # Apply relabeling
    df['new_misconception'] = df.apply(get_new_misconception, axis=1)

    # Filter out high-confidence rows (where new_misconception is None)
    rows_before = len(df)
    df_filtered = df[df['new_misconception'].notna()].copy()
    rows_after = len(df_filtered)
    rows_removed = rows_before - rows_after

    # Update the Misconception column
    df_filtered['Misconception'] = df_filtered['new_misconception']

    # Print summary
    print(f"Relabeling Summary (threshold={threshold}):")
    print(f"\nOriginal 'Other' samples: {rows_before}")
    print(f"  'Other' predictions: {(other_df_with_predictions['top_1_misconception'] == 'Other').sum()}")
    print(f"  'Irrelevant' predictions: {(other_df_with_predictions['top_1_misconception'] == 'Irrelevant').sum()}")
    print(f"  Other misconceptions (low conf <{threshold}): {((other_df_with_predictions['top_1_probability'] < threshold) & (~other_df_with_predictions['top_1_misconception'].isin(['Other', 'Irrelevant']))).sum()}")
    print(f"  Other misconceptions (high conf ≥{threshold}): {rows_removed} [FILTERED OUT]")

    print(f"\nFinal dataset: {rows_after} rows")

    # Count final labels
    all_labels = []
    for labels in df_filtered['Misconception']:
        all_labels.extend(labels)

    from collections import Counter
    label_counts = Counter(all_labels)

    print(f"\nFinal misconception label counts:")
    for label, count in label_counts.most_common():
        print(f"  {label}: {count}")

    # Count multi-label patterns
    print(f"\nFinal label combinations:")
    pattern_counts = Counter([tuple(labels) for labels in df_filtered['Misconception']])
    for pattern, count in pattern_counts.most_common():
        print(f"  {list(pattern)}: {count}")

    return df_filtered

# Usage
other_df_relabeled = relabel_low_confidence_other_samples(
    other_df_with_predictions,
    threshold=0.68
)

# Save results
output_path = os.path.join(checkpoint_dir, 'other_samples_relabeled.csv')
other_df_relabeled.to_csv(output_path, index=False)
print(f"\n💾 Final relabeled 'Other' samples saved to: {output_path}")

# Optionally, also save the high-confidence rows to add back to training
high_conf_df = other_df_with_predictions[
    (other_df_with_predictions['top_1_probability'] >= 0.68) &
    (~other_df_with_predictions['top_1_misconception'].isin(['Other', 'Irrelevant']))
].copy()

if len(high_conf_df) > 0:
    high_conf_df['Misconception'] = high_conf_df['top_1_misconception'].apply(lambda x: [x])
    high_conf_output = os.path.join(data_dir, 'other_samples_promoted_to_training.csv')
    high_conf_df.to_csv(high_conf_output, index=False)
    print(f"\n💾 High-confidence samples for training saved to: {high_conf_output}")
    print(f"   ({len(high_conf_df)} samples to add back to training data)")

,QuestionText,StudentExplanation,MC_Answer,Category,Misconception,Correctness,row_id,problem_type,tuple_key,Mathematical_Context,Original_Explanation,Mathematical_Context_Reduced,top_1_misconception,top_1_probability,top_2_misconception,top_2_probability
0,A bag contains \( 24 \) yellow and green balls...,(using the explanation above) ALL_UNITS divide...,\( 9 \),[False_Neither],[Other],0,[22984],4,(A bag contains \( 24 \) yellow and green ball...,,(using the explanation above) 24 divided by 4 ...,,Wrong_fraction,0.998624,Irrelevant,0.029162
1,A bag contains \( 24 \) yellow and green balls...,. ALL_UNITS- FRACTION_DENOMINATOR=3x3=NUM_TIME...,\( 15 \),[True_Neither],[Other],1,[22014],4,(A bag contains \( 24 \) yellow and green ball...,,. 24- 8=3x3=9 and i thought it was higher i go...,,No Misconception,0.134902,Wrong_Fraction,0.000638
2,A bag contains \( 24 \) yellow and green balls...,FRAC_COUNT eighth is WHOLE_DIV_DEN_SIMP so 6 e...,\( 15 \),[True_Neither],[Other],1,[22017],4,(A bag contains \( 24 \) yellow and green ball...,,1 eighth is 3 so 6 eighths is 15,,No Misconception,0.999613,Positive,0.000091
3,A bag contains \( 24 \) yellow and green balls...,FRAC_COUNT/FRACTION_DENOMINATOR is FRACTION an...,\( 15 \),[True_Neither],[Other],1,[22034],4,(A bag contains \( 24 \) yellow and green ball...,,1/8 is 3/8 and times it times by 5,,No Misconception,0.995283,Wrong_fraction,0.002530
4,A bag contains \( 24 \) yellow and green balls...,FRAC_COUNT/FRACTION_DENOMINATOR of ALL_UNITS i...,\( 9 \),[False_Neither],[Other],0,[22995],4,(A bag contains \( 24 \) yellow and green ball...,,1/8 of 24 is 3 so there are 9 - 3 = 6.,,Wrong_fraction,0.598325,No Misconception,0.049602
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11645,\( \frac{A}{10}=\frac{9}{15} \) What is the va...,you have to make it the lcd and then find t ou...,\( 4 \),[False_Neither],[Other],0,[12509],14,(\( \frac{A}{10}=\frac{9}{15} \) What is the v...,F1_NUM_LT_DEN:TRUE | ANSWER_HAS_FRACTION:FALSE,you have to make it the lcd and then find t ou...,F1_NUM_LT_DEN:TRUE | ANSWER_HAS_FRACTION:FALSE,Additive,0.994003,Adding_terms,0.000977
11646,\( \frac{A}{10}=\frac{9}{15} \) What is the va...,you look at the bottom 2 numbers,\( 6 \),[True_Neither],[Other],1,[14261],14,(\( \frac{A}{10}=\frac{9}{15} \) What is the v...,F1_NUM_LT_DEN:TRUE | ANSWER_HAS_FRACTION:FALSE,you look at the bottom 2 numbers,F1_NUM_LT_DEN:TRUE | ANSWER_HAS_FRACTION:FALSE,WNB,0.209398,Additive,0.099924
11647,\( \frac{A}{10}=\frac{9}{15} \) What is the va...,you make the DEN_LEFT F1_DEN because if you do...,\( 6 \),[True_Neither],[Other],1,[14263],14,(\( \frac{A}{10}=\frac{9}{15} \) What is the v...,F1_NUM_LT_DEN:TRUE | ANSWER_HAS_FRACTION:FALSE,you make the 10 5 because if you do x3 you get...,F1_NUM_LT_DEN:TRUE | ANSWER_HAS_FRACTION:FALSE,Irrelevant,0.049406,No Misconception,0.033864
11648,\( \frac{A}{10}=\frac{9}{15} \) What is the va...,you need too times everything equally to get t...,\( 6 \),[True_Neither],[Other],1,[14266],14,(\( \frac{A}{10}=\frac{9}{15} \) What is the v...,F1_NUM_LT_DEN:TRUE | ANSWER_HAS_FRACTION:FALSE,you need too times everything equally to get t...,F1_NUM_LT_DEN:TRUE | ANSWER_HAS_FRACTION:FALSE,Irrelevant,0.646408,No Misconception,0.036406


In [53]:
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_fscore_support
from torch.utils.data import DataLoader
import os

# ============================================================================
# DUAL FEATURE DATASET CLASS
# ============================================================================

class DualFeatureDataset(torch.utils.data.Dataset):
    """Dataset that returns both full and reduced mathematical context"""

    def __init__(self, df, tokenizer, label_encoder_category, label_encoder_misconception,
                 augmenter, max_length=450, kind='val'):
        self.kind = kind
        self.label_encoder_category = label_encoder_category
        self.label_encoder_misconception = label_encoder_misconception
        self.questions = df['QuestionText'].tolist()
        self.explanations = df['StudentExplanation'].tolist()
        self.correctness = df['Correctness'].tolist()
        self.mc_answer = df['MC_Answer'].tolist()
        self.categories = df['Category'].tolist()
        self.misconceptions = df['Misconception'].tolist()
        self.math_context_full = df['Mathematical_Context'].tolist()
        self.math_context_reduced = df['Mathematical_Context_Reduced'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.augmenter = augmenter

    def __len__(self):
        return len(self.questions)

    def format_math_context_as_text(self, context_string):
        """Convert math context to text format"""
        if not context_string:
            return ""

        parts = [part.strip() for part in context_string.split('|') if part.strip()]
        formatted_features = []

        for part in parts:
            if ':' in part:
                key, value = part.split(':', 1)
                formatted_features.append(f"{key.strip()}:{value.strip()}")

        if not formatted_features:
            return ""

        return " <MATH_CONTEXT> " + " | ".join(formatted_features) + " </MATH_CONTEXT> "

    def __getitem__(self, idx):
        question = self.questions[idx]
        explanation = self.explanations[idx]
        correct_label = self.correctness[idx]
        category_labels = self.categories[idx]
        mc_answer = str(self.mc_answer[idx])
        misconception_label = self.misconceptions[idx]

        # Get both full and reduced contexts
        math_context_full = self.format_math_context_as_text(self.math_context_full[idx])
        math_context_reduced = self.format_math_context_as_text(self.math_context_reduced[idx])

        question_with_context = question + " " + mc_answer

        # Encode category label
        if isinstance(category_labels, list):
            category_labels = category_labels[0]
        category_label = self.label_encoder_category.transform([category_labels])[0]

        # Encode misconception labels
        misconception_multi_label = [0] * len(self.label_encoder_misconception.classes_)
        if isinstance(misconception_label, str):
            misconception_label = [misconception_label]
        for mis in misconception_label:
            if mis in self.label_encoder_misconception.classes_:
                idx_mis = self.label_encoder_misconception.transform([mis])[0]
                misconception_multi_label[idx_mis] = 1

        # Create inputs with FULL context (for correctness model)
        explanation_full = math_context_full + " " + explanation
        encoding_full = self.tokenizer(
            question_with_context,
            explanation_full,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        # Create inputs with REDUCED context (for misconception model)
        explanation_reduced = math_context_reduced + " " + explanation
        encoding_reduced = self.tokenizer(
            question_with_context,
            explanation_reduced,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids_full': encoding_full['input_ids'].squeeze(0),
            'attention_mask_full': encoding_full['attention_mask'].squeeze(0),
            'input_ids_reduced': encoding_reduced['input_ids'].squeeze(0),
            'attention_mask_reduced': encoding_reduced['attention_mask'].squeeze(0),
            'correctness': torch.tensor(correct_label, dtype=torch.long),
            'category': torch.tensor(category_label, dtype=torch.long),
            'misconception': torch.tensor(misconception_multi_label, dtype=torch.float32),
            'question_text': question_with_context,
            'explanation_text': explanation
        }


# ============================================================================
# COMBINED VALIDATION FUNCTION
# ============================================================================

def calculate_combined_validation_metrics(
    correctness_model,
    misconception_model,
    val_loader,
    device,
    label_encoder_category,
    label_encoder_misconception
):
    """
    Run validation using two models and combine predictions with rule-based logic
    """
    correctness_model.eval()
    misconception_model.eval()

    # Storage
    all_correctness_preds = []
    all_correctness_labels = []
    all_category_preds = []
    all_category_labels = []
    all_misconception_preds = []
    all_misconception_labels = []

    print(f"Processing {len(val_loader)} batches with dual models...")

    with torch.no_grad():
        for batch_idx, batch in enumerate(val_loader):
            # ========== CORRECTNESS PREDICTION (Full Features) ==========
            input_ids_full = batch['input_ids_full'].to(device)
            attention_mask_full = batch['attention_mask_full'].to(device)

            logits_corr, _, _ = correctness_model(input_ids_full, attention_mask_full)
            preds_correctness = logits_corr.argmax(dim=-1).cpu().numpy()

            # ========== MISCONCEPTION PREDICTION (Reduced Features) ==========
            input_ids_reduced = batch['input_ids_reduced'].to(device)
            attention_mask_reduced = batch['attention_mask_reduced'].to(device)

            _, _, logits_misc = misconception_model(input_ids_reduced, attention_mask_reduced)
            misconception_probs = torch.sigmoid(logits_misc).cpu().numpy()
            preds_misconception = (misconception_probs > 0.5).astype(int)

            # ========== DERIVE CATEGORY FROM RULES ==========
            batch_size = len(preds_correctness)
            preds_category = np.zeros(batch_size, dtype=int)

            for i in range(batch_size):
                # Get predicted misconception (highest probability)
                misc_idx = np.argmax(misconception_probs[i])
                misc_name = label_encoder_misconception.inverse_transform([misc_idx])[0]

                is_correct = preds_correctness[i] == 1

                # Apply category rules
                if misc_name not in ['No Misconception', 'Other']:
                    # Has a real misconception
                    if is_correct:
                        cat_name = 'True_Misconception'
                    else:
                        cat_name = 'False_Misconception'

                elif misc_name == 'No Misconception':
                    if is_correct:
                        cat_name = 'True_Correct'
                    else:
                        cat_name = 'False_Correct'

                else:  # Other
                    if is_correct:
                        cat_name = 'True_Neither'
                    else:
                        cat_name = 'False_Neither'

                # Encode category
                preds_category[i] = label_encoder_category.transform([cat_name])[0]

            # Store predictions and labels
            correctness_labels = batch['correctness'].cpu().numpy()
            category_labels = batch['category'].cpu().numpy()
            misconception_labels = batch['misconception'].cpu().numpy()

            all_correctness_preds.extend(preds_correctness)
            all_correctness_labels.extend(correctness_labels)
            all_category_preds.extend(preds_category)
            all_category_labels.extend(category_labels)
            all_misconception_preds.extend(preds_misconception)
            all_misconception_labels.extend(misconception_labels)

            if (batch_idx + 1) % 50 == 0:
                print(f"  Processed {batch_idx + 1}/{len(val_loader)} batches")

    # Convert to arrays
    all_correctness_preds = np.array(all_correctness_preds)
    all_correctness_labels = np.array(all_correctness_labels)
    all_category_preds = np.array(all_category_preds)
    all_category_labels = np.array(all_category_labels)
    all_misconception_preds = np.vstack(all_misconception_preds)
    all_misconception_labels = np.vstack(all_misconception_labels)

    print(f"\n✅ Processed all batches")
    print(f"   Total samples: {len(all_correctness_preds)}")

    # ========================================================================
    # CALCULATE METRICS
    # ========================================================================

    # --- CORRECTNESS ---
    correctness_macro_f1 = f1_score(all_correctness_labels, all_correctness_preds,
                                    average='macro', zero_division=0)
    corr_precision, corr_recall, corr_f1, corr_support = precision_recall_fscore_support(
        all_correctness_labels, all_correctness_preds, average=None, zero_division=0
    )

    correctness_df = pd.DataFrame({
        'Class': ['Incorrect', 'Correct'],
        'Precision': corr_precision,
        'Recall': corr_recall,
        'F1': corr_f1,
        'Support': corr_support.astype(int)
    })

    # --- CATEGORY (from combined predictions) ---
    category_macro_f1 = f1_score(all_category_labels, all_category_preds,
                                 average='micro', zero_division=0)
    cat_precision, cat_recall, cat_f1, cat_support = precision_recall_fscore_support(
        all_category_labels, all_category_preds, average=None, zero_division=0
    )

    category_df = pd.DataFrame({
        'Class': label_encoder_category.classes_,
        'Precision': cat_precision,
        'Recall': cat_recall,
        'F1': cat_f1,
        'Support': cat_support.astype(int)
    }).sort_values('F1')

    # --- MISCONCEPTION ---
    misconception_classes = label_encoder_misconception.classes_
    misconception_rows = []

    for i, class_name in enumerate(misconception_classes):
        true_labels = all_misconception_labels[:, i]
        pred_labels = all_misconception_preds[:, i]

        f1 = f1_score(true_labels, pred_labels, zero_division=0)
        support = int(true_labels.sum())

        tp = ((pred_labels == 1) & (true_labels == 1)).sum()
        fp = ((pred_labels == 1) & (true_labels == 0)).sum()
        fn = ((pred_labels == 0) & (true_labels == 1)).sum()

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0

        misconception_rows.append({
            'Class': class_name,
            'Precision': precision,
            'Recall': recall,
            'F1': f1,
            'Support': support
        })

    misconception_df = pd.DataFrame(misconception_rows).sort_values('F1')
    misconception_macro_f1 = misconception_df['F1'].mean()

    # ========================================================================
    # PRINT RESULTS
    # ========================================================================
    print("\n" + "="*80)
    print("COMBINED MODEL VALIDATION METRICS")
    print("="*80)

    print("\n📊 CORRECTNESS (from correctness model with full features)")
    print(f"Macro F1: {correctness_macro_f1:.4f}")
    print("\nPer-Class:")
    print(correctness_df.to_string(index=False))

    print("\n📊 CATEGORY (derived from correctness + misconception rules)")
    print(f"Macro F1: {category_macro_f1:.4f}")
    print("\nPer-Class (sorted by F1):")
    print(category_df.to_string(index=False))

    print("\n📊 MISCONCEPTION (from misconception model with reduced features)")
    print(f"Macro F1: {misconception_macro_f1:.4f}")
    print("\nWorst 10:")
    print(misconception_df.head(10).to_string(index=False))
    print("\nBest 10:")
    print(misconception_df.tail(10).to_string(index=False))

    print("\n📈 OVERALL SUMMARY")
    print(f"Correctness Macro F1:    {correctness_macro_f1:.4f}")
    print(f"Category Micro F1:       {category_macro_f1:.4f}")
    print(f"Misconception Macro F1:  {misconception_macro_f1:.4f}")
    avg_f1 = (correctness_macro_f1 + category_macro_f1 + misconception_macro_f1) / 3
    print(f"Average F1:              {avg_f1:.4f}")
    print("="*80)

    return {
        'correctness': {'macro_f1': correctness_macro_f1, 'df': correctness_df},
        'category': {'macro_f1': category_macro_f1, 'df': category_df},
        'misconception': {'macro_f1': misconception_macro_f1, 'df': misconception_df},
        'avg_f1': avg_f1
    }


# ============================================================================
# MAIN SCRIPT
# ============================================================================

if __name__ == "__main__":
    from transformers import AutoTokenizer, BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

    MODEL_NAME = 'Qwen/Qwen2-7B-Instruct'
    BATCH_SIZE = 16
    CORRECTNESS_CHECKPOINT = 'stage2_explanation_model_dec5_restructure.pt'
    MISCONCEPTION_CHECKPOINT = 'stage2_explanation_model_dec5_MISC_ONLY_SUB_Other.pt'

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}\n")

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.model_max_length = 450

    # ========== LOAD CORRECTNESS MODEL ==========
    print("Loading CORRECTNESS model (full features)...")
    correctness_path = os.path.join(checkpoint_dir, CORRECTNESS_CHECKPOINT)
    correctness_checkpoint = torch.load(correctness_path, map_location=device, weights_only=False)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True
    )

    correctness_model = MultiTaskQwen(
        MODEL_NAME,
        len(label_encoder_category.classes_),
        len(label_encoder_misconception.classes_),
        bnb_config=bnb_config
    )

    correctness_model.qwen = prepare_model_for_kbit_training(correctness_model.qwen)
    lora_config = LoraConfig(
        r=64, lora_alpha=64,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        lora_dropout=0.05, bias="none",
        task_type="FEATURE_EXTRACTION"
    )
    correctness_model.qwen = get_peft_model(correctness_model.qwen, lora_config)
    correctness_model.load_state_dict(correctness_checkpoint['model_state_dict'], strict=False)
    correctness_model.to(device)
    correctness_model.eval()
    print("✅ Correctness model loaded\n")

    # ========== LOAD MISCONCEPTION MODEL ==========
    print("Loading MISCONCEPTION model (reduced features)...")
    misconception_path = os.path.join(checkpoint_dir, MISCONCEPTION_CHECKPOINT)
    misconception_checkpoint = torch.load(misconception_path, map_location=device, weights_only=False)

    misconception_model = MultiTaskQwen(
        MODEL_NAME,
        len(label_encoder_category.classes_),
        len(label_encoder_misconception.classes_),
        bnb_config=bnb_config
    )

    misconception_model.qwen = prepare_model_for_kbit_training(misconception_model.qwen)
    misconception_model.qwen = get_peft_model(misconception_model.qwen, lora_config)
    misconception_model.load_state_dict(misconception_checkpoint['model_state_dict'], strict=False)
    misconception_model.to(device)
    misconception_model.eval()
    print("✅ Misconception model loaded\n")

    # Create validation dataset with dual features


    agg_val_copy=agg_val.copy()
    agg_val_copy['Structural_Key'] = agg_val_copy['StudentExplanation'].apply(create_structural_explanation_key)
    val_sample=pd.DataFrame()
    for state in range(31,32):
      agg_val_copy=agg_val_copy.sample(frac=1,random_state=states[state])
    # Group by the key and the Misconception (in case the same template applies to different problems)
      val_sample = pd.concat([val_sample, agg_val_copy.groupby(['Structural_Key']).first().reset_index()])
    print('Validation Size: ',len(val_sample))
    val_dataset = DualFeatureDataset(
        val_sample,
        tokenizer,
        label_encoder_category,
        label_encoder_misconception,
        augmenter,
        max_length=450,
        kind='val'
    )

    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    print(f"✅ Validation dataset: {len(val_dataset)} samples, {len(val_loader)} batches\n")

    # Run combined validation
    results = calculate_combined_validation_metrics(
        correctness_model=correctness_model,
        misconception_model=misconception_model,
        val_loader=val_loader,
        device=device,
        label_encoder_category=label_encoder_category,
        label_encoder_misconception=label_encoder_misconception
    )

    # Save results
    output_dir = os.path.join(checkpoint_dir, 'combined_validation_results')
    os.makedirs(output_dir, exist_ok=True)

    results['correctness']['df'].to_csv(
        os.path.join(output_dir, 'correctness_metrics.csv'), index=False
    )
    results['category']['df'].to_csv(
        os.path.join(output_dir, 'category_metrics.csv'), index=False
    )
    results['misconception']['df'].to_csv(
        os.path.join(output_dir, 'misconception_metrics.csv'), index=False
    )

    print(f"\n✅ Results saved to {output_dir}")

Using device: cuda

Loading CORRECTNESS model (full features)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2Model LOAD REPORT from: Qwen/Qwen2-7B-Instruct
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Correctness model loaded

Loading MISCONCEPTION model (reduced features)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2Model LOAD REPORT from: Qwen/Qwen2-7B-Instruct
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Misconception model loaded

Validation Size:  19840
✅ Validation dataset: 19840 samples, 1240 batches

Processing 1240 batches with dual models...
  Processed 50/1240 batches
  Processed 100/1240 batches
  Processed 150/1240 batches
  Processed 200/1240 batches
  Processed 250/1240 batches
  Processed 300/1240 batches
  Processed 350/1240 batches
  Processed 400/1240 batches
  Processed 450/1240 batches
  Processed 500/1240 batches
  Processed 550/1240 batches
  Processed 600/1240 batches
  Processed 650/1240 batches
  Processed 700/1240 batches
  Processed 750/1240 batches
  Processed 800/1240 batches
  Processed 850/1240 batches
  Processed 900/1240 batches
  Processed 950/1240 batches
  Processed 1000/1240 batches
  Processed 1050/1240 batches
  Processed 1100/1240 batches
  Processed 1150/1240 batches
  Processed 1200/1240 batches

✅ Processed all batches
   Total samples: 19840

COMBINED MODEL VALIDATION METRICS

📊 CORRECTNESS (from correctness model with full features)
Macro F1

In [43]:
# ============================================================================
# BATCH INFERENCE FUNCTION
# ============================================================================
def run_batch_inference(model, data_loader, device, label_encoder_misconception, df=None):
    """Run inference on multiple samples and return results"""
    model.eval()

    all_results = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(data_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            # Forward pass
            _, _, misconception_logits = model(input_ids, attention_mask)

            # Get misconception probabilities
            misconception_probs = torch.sigmoid(misconception_logits).float().cpu().numpy()

            # Process each sample in batch
            for i in range(len(misconception_probs)):
                probs = misconception_probs[i]
                sample_idx = batch_idx * data_loader.batch_size + i

                # Get top 3 predictions
                top_3_indices = np.argsort(probs)[-3:][::-1]

                result = {
                    'question': batch['question_text'][i],
                    'explanation': batch['explanation_text'][i],
                    'Original_Explanation': df.iloc[sample_idx]['Original_Explanation'] if df is not None else 'N/A',
                    'answer': df.iloc[sample_idx]['MC_Answer'] if df is not None else 'N/A',
                    'ground_truth_misc': df.iloc[sample_idx]['Misconception'] if df is not None else 'N/A',
                    'top_3_predictions': []
                }

                for idx in top_3_indices:
                    misc_name = label_encoder_misconception.inverse_transform([idx])[0]
                    prob = probs[idx]
                    result['top_3_predictions'].append({
                        'misconception': misc_name,
                        'probability': prob
                    })

                all_results.append(result)

    return all_results

def print_inference_results(results):
    """Pretty print inference results"""
    for i, result in enumerate(results, 1):
        print("=" * 80)
        print(f"SAMPLE {i}")
        print("=" * 80)

        print(f"\nQuestion: {result['question']}")
        print(f'\nOrginal Explanation: {result['Original_Explanation']}')
        print(f"\nProcessed Input: {result['explanation']}")
        print(f"\nStudent Answer: {result['answer']}")
        print(f"\nGround Truth: {result['ground_truth_misc']}")

        print("\nTop 3 Misconception Predictions:")
        for j, pred in enumerate(result['top_3_predictions'], 1):
            print(f"  {j}. {pred['misconception']}: {pred['probability']:.4f}")
        print()

    print("=" * 80)

# ============================================================================
# MAIN BATCH INFERENCE SCRIPT
# ============================================================================

def run_inference_on_samples(test_samples, label_encoder_category, label_encoder_misconception,
                             augmenter, checkpoint_dir, batch_size=8):
    """
    Run inference on multiple test samples.

    Args:
        test_samples: List of dictionaries or DataFrame with samples
        label_encoder_category: Fitted LabelEncoder for categories
        label_encoder_misconception: Fitted LabelEncoder for misconceptions
        augmenter: Text augmentation object
        checkpoint_dir: Directory containing the checkpoint file
        batch_size: Batch size for inference
    """

    # Configuration
    MODEL_NAME = 'Qwen/Qwen2-7B-Instruct'
    CHECKPOINT_FNAME = 'stage2_explanation_model_dec5_MISC_ONLY_SUB_Other.pt'
    checkpoint_path = os.path.join(checkpoint_dir, CHECKPOINT_FNAME)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Load checkpoint
    print(f"\n🔄 Loading checkpoint from: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    # Get model configuration
    num_categories = len(label_encoder_category.classes_)
    num_misc_classes = len(label_encoder_misconception.classes_)

    print(f"📊 Model Configuration:")
    print(f"  - Categories: {num_categories}")
    print(f"  - Misconceptions: {num_misc_classes}")

    # Define BitsAndBytes Config
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True
    )

    # Initialize model
    print(f"\n🏗️ Initializing model...")
    model = MultiTaskQwen(MODEL_NAME, num_categories, num_misc_classes, bnb_config=bnb_config)

    # Apply LoRA
    model.qwen = prepare_model_for_kbit_training(model.qwen)
    lora_config = LoraConfig(
        r=64, lora_alpha=64,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        lora_dropout=0.05, bias="none",
        task_type="FEATURE_EXTRACTION"
    )
    model.qwen = get_peft_model(model.qwen, lora_config)

    # Load weights
    model.load_state_dict(checkpoint['model_state_dict'], strict=False)
    model.to(device)
    model.eval()
    print("✅ Model loaded successfully")

    # Setup tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.model_max_length = 450

    # Convert to DataFrame if list
    if isinstance(test_samples, list):
        test_df = pd.DataFrame(test_samples)
    else:
        test_df = test_samples.copy()

    print(f"\n📊 Processing {len(test_df)} samples...")
    test_df['MC_Answer'] = test_df['MC_Answer'].apply(pf.standardize_answer_format_v5)
    # Preprocessing
    print(f"🧮 Preprocessing...")
    test_df['Mathematical_Context'] = test_df[['QuestionText', 'MC_Answer', 'problem_type']].apply(
        lambda x: augment_functions.create_generic_math_context(
            x['QuestionText'], x['MC_Answer'], x['problem_type']
        ), axis=1
    )

    test_df['StudentExplanation'] = test_df['StudentExplanation'].apply(augmenter.clean)
    test_df['QuestionText'] = test_df['QuestionText'].apply(pf.strip_extra_whitespace)
    test_df['Original_Explanation']=test_df['StudentExplanation']
    test_df['StudentExplanation'] = test_df.apply(apply_feature_substitution, axis=1)
    test_df['QuestionText'] = test_df.apply(
        lambda x: pf.rephrase_question_and_explanation(x['QuestionText'])
        if x['problem_type'] == 7 else x['QuestionText'], axis=1
    )
    test_df['Mathematical_Context'] = test_df['Mathematical_Context'].apply(
        filter_math_context_case_insensitive
    )

    # Create Dataset
    test_dataset = MultiTaskDatasetTextAugmentedB(
        df=test_df,
        tokenizer=tokenizer,
        label_encoder_category=label_encoder_category,
        label_encoder_misconception=label_encoder_misconception,
        augmenter=augmenter,
        max_length=450,
        kind='val',
        task='misconception'
    )

    test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Run inference
    print(f"\n🚀 Running inference...\n")
    results = run_batch_inference(
        model, test_dataloader, device,
        label_encoder_misconception, df=test_df
    )

    # Print results
    print_inference_results(results)

    return results

# ============================================================================
# USAGE
# ============================================================================

if __name__ == "__main__":
    # Your test samples
    test_samples = []
    question_text= "Find \\(\\frac{3}{18} \\times 2\\). simplify if necessary."#"Calculate \\(\\frac{3}{15} \\times 8\\). reduce if necessary."
    # NO MISCONCEPTION
    test_sample = {
        "QuestionText": question_text,
        "StudentExplanation": "I did 2 times 3 over 18. so its 6/18 which simplifies to 1/3",
        "Category": ['False_Misconception'],
        "Misconception": ['No Misconception'],
        "MC_Answer": "\\( \\frac{1}{3} \\)",
        'Correctness': 1,
        'problem_type': 3
    }
    test_samples.append(test_sample)

    # INCOMPLETE
    test_sample = {
        "QuestionText": question_text,
        "StudentExplanation": "I did 2 times 3 over 18. so its 6/18",
        "Category": ['False_Misconception'],
        "Misconception": ['Incomplete'],
        "MC_Answer": "\\( \\frac{6}{18} \\)",
        'Correctness': 0,
        'problem_type': 3
    }
    test_samples.append(test_sample)

    # INVERSION
    test_sample = {
        "QuestionText": question_text,
        "StudentExplanation": "I inverted the 2 to make it 3/18 times 1/2. 3x1=3 and 18x2=36, so its 2/36 which simplifies to 1/18",
        "Category": ['False_Misconception'],
        "Misconception": ['Inversion'],
        "MC_Answer": "\\( \\frac{1}{18} \\)",
        'Correctness': 0,
        'problem_type': 3
    }
    test_samples.append(test_sample)

    # INVERSION + INCOMPLETE
    test_sample = {
        "QuestionText": question_text,
        "StudentExplanation": "I inverted the 2 to make it 3/18 times 1/2. 3x1=3 and 18x2=36, so its 3/36",
        "Category": ['False_Misconception'],
        "Misconception": ['Inversion', 'Incomplete'],
        "MC_Answer": "\\( \\frac{3}{36} \\)",
        'Correctness': 0,
        'problem_type': 3
    }
    test_samples.append(test_sample)

    # DUPLICATION
    test_sample = {
        "QuestionText": question_text,
        "StudentExplanation": "I multiplied the 2 to the 3 and to the 18. so 6/36, this reduces to 1/6",
        "Category": ['False_Misconception'],
        "Misconception": ['Duplication'],
        "MC_Answer": "\\( \\frac{1}{6} \\)",
        'Correctness': 0,
        'problem_type': 3
    }
    test_samples.append(test_sample)

    # DUPLICATION + INCOMPLETE
    test_sample = {
        "QuestionText": question_text,
        "StudentExplanation": "I multiplied the 2 to the 3 and to the 18. so 6/36",
        "Category": ['False_Misconception'],
        "Misconception": ['Duplication', 'Incomplete'],
        "MC_Answer": "\\( \\frac{6}{36} \\)",
        'Correctness': 0,
        'problem_type': 3
    }
    test_samples.append(test_sample)



    # WRONG_OPERATION
    test_sample = {
        "QuestionText": question_text,
        "StudentExplanation": "I combined the whole number with the fraction to make mixed fraction 2 and 3/18, i reduced the 3/18 to 1/6 so its 2 1/6",
        "Category": ['False_Misconception'],
        "Misconception": ['Wrong_Operation'],
        "MC_Answer": "\\( 2 \\frac{1}{6} \\)",
        'Correctness': 0,
        'problem_type': 3
    }
    test_samples.append(test_sample)


    # Run inference
    results = run_inference_on_samples(
        test_samples=test_samples,
        label_encoder_category=label_encoder_category,
        label_encoder_misconception=label_encoder_misconception,
        augmenter=augmenter,
        checkpoint_dir=checkpoint_dir,
        batch_size=8
    )


Using device: cuda

🔄 Loading checkpoint from: /content/drive/MyDrive/LLM_Misconception_Project/checkpoints/stage2_explanation_model_dec5_MISC_ONLY_SUB_Other.pt
📊 Model Configuration:
  - Categories: 6
  - Misconceptions: 37

🏗️ Initializing model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2Model LOAD REPORT from: Qwen/Qwen2-7B-Instruct
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded successfully

📊 Processing 7 samples...
🧮 Preprocessing...

🚀 Running inference...

SAMPLE 1

Question: Find \(\frac{3}{18} \times 2\). simplify if necessary. \( \frac{1}{3} \)

Orginal Explanation: I did 2 times 3/18. so its 6/18 which simplifies to 1/3

Processed Input:  <MATH_CONTEXT> F1_NUM_LT_DEN:TRUE | ANSWER_HAS_FRACTION:TRUE | ANSWER_FRAC_IS_SIMPLIFIED:True </MATH_CONTEXT>  I did WHOLE_NUMBER times F1_BOTH_MULT_N2. so its F1xN1_CORRECT_UNSIM which simplifies to F1xN1_SIM

Student Answer: \( \frac{1}{3} \)

Ground Truth: ['No Misconception']

Top 3 Misconception Predictions:
  1. No Misconception: 0.9988
  2. Duplication: 0.0000
  3. Inversion: 0.0000

SAMPLE 2

Question: Find \(\frac{3}{18} \times 2\). simplify if necessary. \( \frac{6}{18} \)

Orginal Explanation: I did 2 times 3/18. so its 6/18

Processed Input:  <MATH_CONTEXT> F1_NUM_LT_DEN:TRUE | ANSWER_HAS_FRACTION:TRUE | ANSWER_FRAC_IS_SIMPLIFIED:False </MATH_CONTEXT>  I did WHOLE_NUMBER times F1_BOTH_MULT_

In [46]:
check=pd.read_csv(os.path.join(data_dir,'Assessment_Wrong_Operation_Val_1.csv'))
check['Correctness']=0 #check['Category'].apply(lambda x: )
check['Category']=check['Category'].apply(ast.literal_eval)
check['Misconception']=check['Misconception'].apply(ast.literal_eval)
check
results = run_inference_on_samples(
        test_samples=check,
        label_encoder_category=label_encoder_category,
        label_encoder_misconception=label_encoder_misconception,
        augmenter=augmenter,
        checkpoint_dir=checkpoint_dir,
        batch_size=8
    )

Using device: cuda

🔄 Loading checkpoint from: /content/drive/MyDrive/LLM_Misconception_Project/checkpoints/stage2_explanation_model_dec5_MISC_ONLY_SUB_Other.pt
📊 Model Configuration:
  - Categories: 6
  - Misconceptions: 37

🏗️ Initializing model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2Model LOAD REPORT from: Qwen/Qwen2-7B-Instruct
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded successfully

📊 Processing 4 samples...
🧮 Preprocessing...

🚀 Running inference...

SAMPLE 1

Question: Compute \(\frac{1}{6} \times 12\). reduce if necessary. \( 12 \frac{1}{6} \)

Student Explanation:  <MATH_CONTEXT> F1_NUM_LT_DEN:TRUE | ANSWER_HAS_FRACTION:TRUE | ANSWER_FRAC_IS_SIMPLIFIED:True </MATH_CONTEXT>  I put the F1 with the F1_NUM_TIMES_WHOLE to make F1_NUM_TIMES_WHOLE and F1

Student Answer: \( 12 \frac{1}{6} \)

Ground Truth: ['Wrong_Operation']

Top 3 Misconception Predictions:
  1. Wrong_Operation: 0.9998
  2. Inversion: 0.0079
  3. Irrelevant: 0.0001

SAMPLE 2

Question: Find \(\frac{3}{18} \times 2\). simplify if necessary. \( 2 \frac{1}{6} \)

Student Explanation:  <MATH_CONTEXT> F1_NUM_LT_DEN:TRUE | ANSWER_HAS_FRACTION:TRUE | ANSWER_FRAC_IS_SIMPLIFIED:True </MATH_CONTEXT>  I combined the whole number with the fraction to make mixed fraction WHOLE_NUMBER and F1_BOTH_MULT_N2, i reduced the F1_BOTH_MULT_N2 to F1 so its WRONG_OPERATION_ANSWER_SIMP

Student